# **CardiAg: Respiratory Phase Analysis**

Pipeline for the analysis of behavioral events from the CardiAg Intentional Binding task with respect to respiratory phase derived from the respiration belt (RESP) signal, including circular and binary analysis of respiratory phase. 

Authors: Marta Gerosa

Created on: 27 May 2025

Last update (by M. Gerosa): 21 April 2026

Before using the pipeline, we recommend downloading the already preprocessed RESP files and store them in the `data/derivatives/resp-preproc` folder. Otherwise, if you wish to preprocess your own raw RESP data, you can run the `CardiAg_resp_preproc.ipynb` on the subject-level BIDS-compatible `*_physio.tsv.gz` files. 

The following packages are required: `rpy2` (>= 3.5). Importantly, a working installation of R is also required to run the circular analysis section. 

## **Pipeline structure**

The following steps are included:

1. **Behavioral data import and merge into long format**: load the preprocessed behavioral data (`_beh-preproc.tsv`) from the `/derivatives/beh-preproc/sub-xx/` folder for each subject and merge all of them into one DataFrame `behpreproc_long` in long format. Please note that the behavioral data file should have one row per each trial, containing task-relevant information (e.g., condition, n trial etc.) and - at least - one column containing timestamps (in sec) for a task event of interest (e.g., keypress or visual stimulus onset). 
2. **Preprocessed RESP data import and conversion**: load the preprocessed RESP data (`_resp-preproc.json`) from the `/derivatives/resp-preproc/sub-xx/` folder and its metadata (`_physio.json`) from the BIDS raw data folder for each subject. Extract the sample idx for expiration onsets (peaks) & inspiration onsets (troughs), then convert them into timestamps (in sec), as well as load data from the corresponding phase vector (`*_resp-phase-vector.tsv.gz`) (expiration: 0 -> π; inspiration: π -> 2π). 
3. **RESP features extraction around task event**: extract the RESP features around a (user-defined) task event, mainly expiration onsets (peaks) and inspiration onsets (troughs) needed for segmentation into expiration and inspiration, on a trial-by-trial basis. These features are used to calculate the duration of the respiratory cycle around the task events, the instantaneous respiratory rate (RR_resp), the temporal and angular position of the task event with respect to one respiratory cycle (for circular analysis), the duration of expiration and inspiration, as well as for binning events into expiration or inspiration (for binary analysis). If `outlier_remove` is `True`, outliers in expiration or inspiration duration that are +/- 4 SD from the individual mean are removed (in line with Engelen et al., 2024). Furthermore, descriptives for the individual respiratory phase durations are provided. The final dataset comprising behavioral data and corresponding RESP features and individual expiration/inspiration segmentation are stored in the `task-CardiAgIBTask_beh_resp_action.tsv` and `task-CardiAgIBTask_beh_resp_tone.tsv` for action and tone trials, respectively. This corresponds to **Supplementary Results sections 1.2.3**, with the respective **Figure S3b**.
4. **Binary analysis - expiration vs. inspiration ratio of task event [Results 2.3.2]**: perform binary analysis of individual respiratory phases (expiration vs. inspiration) with respect to a task event, by computing the phase-specific proportion of task events, relative to all trials, normalized by the proportion of the respective respiratory phase in the entire respiratory cycle. In group-level analysis, normalized expiratory and inspiratory ratios are tested using a two-sided paired t-test. If a respiratory effect is present, the phase-specific ratio value is expected to be > 1, indicating over-proportional accumulation of task events in the respective phase. This corresponds to **Results section 2.3.2** with corresponding **Figure 4b**. 
5. **Binary analysis - respiratory phase binning of intentional binding [Results 2.3.3 & 2.5.3]**: compute respiratory phase differences in action binding or tone binding (i.e., dependent variable), respectively, by binning action or tone into expiration/inspiration; this corresponds to **Results section 2.3.3** with corresponding **Figures 4c, 4d and 4e**. In addition, perfom correlation analysis between the individual normalized expiratory and inspiratory ratios for action/tone trials and the corresponding intentional binding measure (i.e., action binding and tone binding, respectively); this corresponds to **Results section 2.5.3**, as well as **Figures 7c & 7d**. The participant-level means of action/tone binding binned by respiratory phase are saved under `task-CardiAgIBTask_resp_actbinding_avg.tsv` and `task-CardiAgIBTask_resp_tonebinding_avg.tsv`. 
6. **Circular analysis of task events across respiratory cycle [Results 2.3.1]**: first, perform 1st-level circular analysis (within-subject) of task event (i.e., action onset) across respiratory cycle, testing the individual circular means against null distribution using participant-level Rayleigh test. Second, perform 2nd-level circular analysis (between-subject) of task event (i.e., action onset) across respiratory cycle, testing the group-level mean resultant length ϱ against the null hypothesis of uniform distribution using a Rayleigh test. To calculate 95% confidence intervals and significance, a non-parametric bootstrap analysis (10000 iterations, with replacement) is performed. The resulting group-level circular plot with 2nd-level circular mean and mean resultant length, individual circular means, average circular distribution with 95% CI and significant segments, as determined by bootstrap analysis, is saved. This corresponds to **Results section 2.3.1** with **Figure 4a**. Lastly, condition-wise pairwise comparisons of circular distributions of action onsets according to intentional binding condition are computed, as corresponding to **Extended Data Figure 2**.
7. **Pre- and post-event respiratory cycle changes [Results 2.5.2]**: analyze changes in the duration of respiratory cycles before and after the action onset, indicative of respiratory deceleration and/or acceleration. This corresponds to **Results section 2.5.2**, with **Extended Data Figure 3**. 

In [4]:
############## Import modules ##############

import pandas as pd
import numpy as np
import os
import json
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from IPython.display import Markdown as md
from tqdm import tqdm
import pickle
import pingouin as pg
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import fdrcorrection
import gzip
import matplotlib.patches as patches

In [5]:
############## Settings: optional pipeline steps ##############

# For all sections: choose the task event type that will be analysed with respect to the respiratory cycle,
# by specifying the column name with timestamps (in sec) and the abbreviation to be appended in the variable names

# Action onset analysis
event_col = 'keypress_onset'    # define column of beh dataset with timestamps (in sec) of task event onset
abbrev = 'act'                  # define abbreviation of task event for further column names in respiratory phase analysis

# Tone onset analysis
event_col_tone = 'tone_onset'    # define column of beh dataset with timestamps (in sec) of task event onset
abbrev_tone = 'tone'             # define abbreviation of task event for further column names in respiratory phase analysis

# For all sections: specify your column names for participant ID, condition, trial and block (the latter is optional)
column_map = {
    'participant': 'subjID',
    'condition': 'condition',
    'trial': 'n_trial',
    'block': 'n_block' # remove this key-value pair if block is not available
}

## **Settings: R-Python interface**

Please note that, in order to use this pipeline (in particular, to conduct circular analysis of cardiac phase), a working installation of R is needed (>= 4.0). In the section below, the user should specify the path to their R installation and their rpy2 installation - the former will likely be under the `Program Files/R` folder, the latter under the dedicated virtual env folder (check under `Lib/site-packages/`). 

* When using the pipeline for the first time, set `rinstall_firsttime` to `True`. Otherwise, keep it as `False`. 

Further information on the rpy2 module and how to interface with R in Python can be found here: https://rpy2.github.io/

In [6]:
############## Import modules to interact with R packages in Python ##############

# Path to user-defined R installation
os.environ['R_HOME'] = 'C:/Program Files/R/R-4.4.0' # define path to R
os.environ['R_USER'] = 'C:/Users/.../AppData/Local/miniforge3/envs/cardiag_env/Lib/site-packages/rpy2' # define path to rpy2 in your virtual environment

# Set to TRUE the first time that the script is run to download the necessary R packages, otherwise FALSE
rinstall_firsttime = False


# Import or install R packages that will be used 
from rpy2.robjects.packages import importr
import rpy2.robjects.packages as rpackages
from rpy2.robjects.vectors import StrVector
from rpy2.robjects import pandas2ri
import rpy2.robjects as ro
from rpy2 import robjects

# Activate pandas-R conversion
pandas2ri.activate()

# Import useful R packages
utils = rpackages.importr('utils') # import R 'utils' package
grdevices = importr('grDevices')
graphics = importr('graphics')
lme4 = importr('lme4')
base = importr('base')

if rinstall_firsttime:
    utils.chooseCRANmirror(ind=1) # Select the first mirror for R packages
    packnames = ('circular') # R packages to install/import
    # Selectively install what needs to be installed
    names_to_install = [x for x in packnames if not rpackages.isinstalled(x)]
    if len(names_to_install) > 0:
        utils.install_packages(StrVector(names_to_install))

# Import "circular" package
circular = rpackages.importr('circular')

## **1. Behavioral data import and merge into long format**

This section imports the preprocessed behavioral data `_beh-preproc.tsv` for each participant from the `/derivatives/beh-preproc` folder, turning them into a DataFrame (`subj_behpreproc`). Then, all the participant-specific DataFrames are concatenated together into the `behpreproc_long` DataFrame in long format, i.e. one row per trial, all participants into one file. In detail: 
- Iterate through participant IDs specified in a list.
- Define the name and directory path for the `_beh-preproc.tsv` file containing preprocessed behavioral data for each participant. 
- Import the TSV file and store as a DataFrame (`subj_behpreproc`).
- Merge the participant-specific `subj_behpreproc` DataFrames into one unique `behpreproc_long` DataFrame in long format. 
- Define the base directory `derivatives/resp-phase-analysis` folder for storage of results.

In [ ]:
################## 1. Behavioral data import and merge into long format ##################

# Sub-16 excluded due to extreme JE distribution, sub-14 excluded due to missing HW triggers
# Sub-33 excluded due to outlier IB measures (similar to ECG analysis)
participant_ids = [1, 2, 3, 5, 6, 12, 13, 15, 17, 18, 19, 
                   20, 21, 22, 23, 24, 25, 26, 27, 28, 30,      # removed: sub-29 (bad signal quality)
                   31, 32, 34, 35, 36, 37, 38, 39, 41,          # removed: sub-42 (bad signal quality)
                   43, 44, 45, 47, 48, 49, 51, 53, 54, 55, 57]  # removed: sub-46 (bad signal quality)

# Specify the data path info
wd = r'.\data'                  # directory of data storage
exp_name = 'CardiAgIBTask' 
datatype_name = 'beh'           # current datatype folder according to BIDS

# Initialize empty list to store dfs with beh preprocessed data for all participants
subj_dfs = []

# Iterate through each participant
for subj in participant_ids:

    subj_id = 'sub-' + str(subj) # participant ID (in BIDS format)
    behpreproc_fname = f'{subj_id}_task-{exp_name}_beh-preproc.tsv'

    # Merge information into complete datapath
    behpreproc_dir = os.path.join(wd, 'derivatives', 'beh-preproc', subj_id, 
                                  datatype_name, behpreproc_fname)

    # Read TSV file into a dataframe
    subj_behpreproc = pd.read_csv(behpreproc_dir, sep='\t', header=0)

    # Reset the index to avoid having duplicate index columns and append to list
    subj_behpreproc.reset_index(drop=True, inplace=True)
    subj_dfs.append(subj_behpreproc)

# Create a long data frame with beh preprocessed data from all participants
behpreproc_long = pd.concat(subj_dfs, ignore_index=True)
behpreproc_long.rename(columns={'Unnamed: 0': 'subj_index'}, inplace=True) # comment if unnecessary

# Subselect only relevant behavioral data for respiratory-beh analysis
behpreproc_long = behpreproc_long[['subjID', 'condition', 'n_block', 'n_trial', 
                                   'real_report_diff_act_deg', 'real_report_diff_tone_deg', 'JE_act', 'JE_tone', 
                                   'trial_onset', 'keypress_onset', 'tone_onset']].copy()


####### Create general directory for saving output from respiratory phase analysis #######

# Check if the BIDS directory 'derivatives/resp-phase-analysis/' exists, if not create it
respphase_folder = 'resp-phase-analysis'
respphase_dir = os.path.join(wd, 'derivatives', respphase_folder)
if not os.path.exists(respphase_dir):
    os.makedirs(respphase_dir)

## **2. Preprocessed RESP data import and conversion**

This section import the preprocessed RESP data (`_resp-preproc.json`) from the `/derivatives/resp-preproc/sub-xx/` folder, as well as the corresponding physio metadata (`_physio.json`) from the `rawdata` folder, in order to extract the sample idx for expiration peaks & inspiration troughs and covert them into timestamps (in sec). The following RESP features are derived from the RESP preprocessing file: expiration onsets (peaks), inspiration onsets (troughs). Furthermore, data from the corresponding phase vector (`*_resp-phase-vector.tsv.gz`) stored in `derivatives/resp-preproc/sub-XX/beh/` folder is loaded (expiration: 0 -> π; inspiration: π -> 2π). 

In [ ]:
############## 2. Preprocessed RESP data import and conversion ##############

# Initialize empty dict to store corrected rpeaks idx for all participants
subj_resp = {}

# Iterate through each participant
for subj in participant_ids:

    subj_id = 'sub-' + str(subj)  # Participant ID (in BIDS format)

    ############## Load RESP signal metadata ##############

    # Specify the name and directory of physio JSON file (for metadata)
    json_md_fname = f'{subj_id}_task-{exp_name}_physio.json'
    json_md_dir = os.path.join(wd, subj_id, datatype_name, json_md_fname)

    # Extract physio metadata from JSON file
    fjson = open(json_md_dir)
    resp_metadata = json.load(fjson) # load metadata from _physio.json
    sfreq = resp_metadata['SamplingFrequency'] # get sampling rate frequency

    ############## Load preprocessed RESP data (incl. exp & insp onsets, resp cycles) ##############

    json_resp_fname = f'{subj_id}_task-{exp_name}_resp-preproc.json'
    json_resp_dir = os.path.join(wd, 'derivatives', 'resp-preproc', 
                                 subj_id, datatype_name, json_resp_fname)

    # Check if JSON file exists and, if so, load it
    if os.path.exists(json_resp_dir):
        with open(json_resp_dir, 'r') as json_resp_preproc:
            resp_preproc_data = json.load(json_resp_preproc)

        ####### Extract RESP features idx #######

        # Extract RESP features: expiration onsets and inspiration onsets
        exp_onsets_idx = resp_preproc_data['exp_onsets']['ManualCorr']      # prefer manually corrected data
        insp_onsets_idx = resp_preproc_data['insp_onsets']['ManualCorr']    # prefer manually corrected data

        # Remove first inspiration if it's None, and align arrays
        if insp_onsets_idx[0] is None:
            insp_onsets_idx = insp_onsets_idx[1:]
            exp_onsets_idx = exp_onsets_idx[:-1]

        subj_resp[subj_id] = {'exp_onsets_idx': exp_onsets_idx,     # sample idx of expiration peaks
                              'insp_onsets_idx': insp_onsets_idx}   # sample idx of inspiration troughs
        
        ####### Transform idx of RESP features to timestamps #######

        # Transform idx into timestamps (in sec)
        exp_onsets_time = [idx / sfreq for idx in exp_onsets_idx if idx is not None]
        insp_onsets_time = [idx / sfreq for idx in insp_onsets_idx if idx is not None]

        subj_resp[subj_id]['exp_onsets_s'] = exp_onsets_time        # timestamps of expiration peaks
        subj_resp[subj_id]['insp_onsets_s'] = insp_onsets_time      # timestamps of inspiration troughs

    else:
        print(f"RESP preprocessing JSON file not found for {subj_id}")
    

    ####### Get the respiratory phase vector (exp: 0 -> π; insp: π -> 2π) #######

    # Directory of respiratory phase vector TSV.GZ file in derivatives/resp-preproc/sub-XX/beh/ folder 
    phasevec_fname = f'{subj_id}_task-{exp_name}_resp-phase-vector.tsv.gz'
    phasevec_fpath = os.path.join(wd,'derivatives', 'resp-preproc', 
                                subj_id, datatype_name, phasevec_fname)
    
    # Check if TSV.GZ file exists and, if so, load the associated df
    if os.path.exists(phasevec_fpath):
        resp_phase_vec_df = pd.read_csv(phasevec_fpath, sep='\t', header=0, compression='gzip', usecols=['phase_vector_manualcorr'])

        # Extract only the respiratory phase vector for manually corrected peaks/troughs
        resp_phase_vec = resp_phase_vec_df['phase_vector_manualcorr'].to_list()
    
        # Store the phase vector
        subj_resp[subj_id]['resp_phase_vec'] = resp_phase_vec
    
    else:
        print(f"RESP phase vector TSV.GZ file not found for {subj_id}")

## **3. RESP features extraction around task event**

This section extracts the RESP features around a (user-defined) task event, mainly expiration onsets (peaks) and inspiration onsets (troughs) needed for segmentation into expiration and inspiration, on a trial-by-trial basis. Then, proceeds to calculate the duration of the respiratory cycle around the task events, the instantaneous respiratory rate (RR_resp), the temporal and angular position of the task event with respect to one respiratory cycle (for circular analysis), the duration of expiration and inspiration, and lastly determines whether the event falls into expiration or inspiration (for binary analysis). If `outlier_remove` is `True`, outliers in expiration or inspiration duration that are +/- 4 SD from the individual mean are removed. In detail: 

- The user must specify the column name contaning timestamps - in sec - for the task event, using the function argument `event_col` (e.g., 'keypress_onset'), as well as an abbreviated name for the event type (e.g., 'act')
- For each trial, the relevant ECG features surrounding a user-defined task event of interest are extracted: expiration onsets preceding (ExpOn_pre) and following (ExpOn_post) the task event, and inspiration onset comprised between them (InspOn). 
- These RESP features are used to determine on a trial-by-trial basis: respiratory cycle duration (in sec), instantaneous respiratory rate (RR_resp; in bpm and Hz), time difference of the task event with respect to the ExpOn_pre, angular difference (in rad) of the task event with respect to ExpOn_pre (for circular analysis; task events between ExpOn_pre and InspOn take values between 0 and pi, whilst those between InspOn and ExpOn_post take values between pi and 2pi). 
- Lastly, the respiratory phases (expiration, inspiration) are segmented, deriving their relative duration and determining in which of the two phases the task event falls, by assigning a binary 0/1 value. 
- If `outlier_remove` is `True`, outliers that exceed +/- 4 SD from the individual mean in expiration duration or inspiration duration are removed, and the corresponding RESP features are set as `NaN`. The descriptives regarding the outlier removal procedure (e.g., average number and percentage of outliers excluded per event type, across participants) are saved in `outliers_df`.

### **3a. RESP features extraction around task event**

In [9]:
############## 3a. RESP features extraction around task event ##############

# Define a custom function for RESP features extraction around task event of interest

def extract_resp_features_event(event_col, abbrev, behpreproc_long=behpreproc_long, subj_resp=subj_resp, 
                                outlier_remove=True, participant_col=column_map['participant']):
    """
    Extract RESP features around a specified event of interest and create a preprocessed df 
    per trial and participants, used in later respiratory phase analysis. 

    Parameters
    -----------
    event_col : str
        Column name from `behpreproc_long` containing the timestamps in sec for the event of interest
        (e.g., 'keypress_onset'), on which cardiac phase analysis will be conducted. 
    abbrev : str
        Abbreviated name for the event type (e.g., 'act'), used as suffix for all the derived columns.
    behpreproc_long : DataFrame
        DataFrame containing behavioral data of all participants in long format (one row per trial) 
        and events timestamps, created in earlier steps. Default is `behpreproc_long`. 
    subj_resp : dict
        Dictionary of ECG features in idx and sec per participant, created in earlier steps. Default is `subj_resp`. 
    outlier_remove : bool, default True
        If True, remove outliers in expiration or inspiration duration that are +/- 4 SD from the individual mean.
    participant_col : str
        Column name storing the participant IDs.   

    Returns
    --------
    DataFrame:
        Concatenated DataFrame containing the original behavioral data for all participants plus columns 
        with ECG features relative to the user-defined task event, including:
            - `ExpOn_pre_{abbrev}`: timestamp (in sec) of expiration onset pre (i.e., just before task event)
            - `ExpOn_post_{abbrev}`: timestamp (in sec)  of expiration onset post (i.e., just after task event)
            - `InspOn_{abbrev}`: timestamp (in sec) of inspiration onset between ExpOn_pre and ExpOn_post
            - `resp_cycle_dur_{abbrev}`: respiratory cycle duration around event onset (in sec) as distance between ExpOn_pre and ExpOn_post
            - `RR_resp_1perMin_{abbrev}`: instantaneous respiratory rate (RR_resp) per min (in breaths per minute - bpm) based on respiratory cycle length
            - `RR_resp_1perSec_{abbrev}`: instantaneous respiratory rate (RR_resp) per sec (in Hz) based on respiratory cycle length
            - `diff_event_ExpOn_{abbrev}`: time difference (in sec) from task event to the preceding expiration onset
            - `event_resp_rad_{abbrev}`: angular position of task event (in radians) relative to respiratory cycle
            - `exp_dur_{abbrev}`: total expiration duration (in sec), calculated expiration onset (ExpOn_pre) to following inspiration onset (InspOn)
            - `insp_dur_{abbrev}`: duration of systolic PEP (in sec), calculated from inspiration onset (InspOn) to following expiration onset (ExpOn_post)
            - `{abbrev}_exp`: binary value 0/1 of whether the task event falls into expiration window
            - `{abbrev}_insp`: binary value 0/1 of whether the task event falls into inspiration window

    """
    
    beh_resp_dfs = [] # initialize empty list to store dfs for all participants

    # Extract relevant RESP features for the current participant
    for subj in behpreproc_long[participant_col].unique():
        subj_resp_data = subj_resp.get(f'sub-{subj}', {})
        exp_onsets_s = subj_resp_data.get('exp_onsets_s', [])
        insp_onsets_s = subj_resp_data.get('insp_onsets_s', [])

        # Filter df with preprocessed beh data for current participant
        subj_beh_resp = behpreproc_long[behpreproc_long[participant_col] == subj].copy()

        # Specify relevant column names for resp-beh analysis based on the event abbreviation
        columns = {
            'ExpOn_pre': f'ExpOn_pre_{abbrev}', 'ExpOn_post': f'ExpOn_post_{abbrev}',
            'InspOn': f'InspOn_{abbrev}', 'resp_cycle_dur': f'resp_cycle_dur_{abbrev}',
            'RR_resp_1perMin': f'RR_resp_1perMin_{abbrev}', 'RR_resp_1perSec': f'RR_resp_1perSec_{abbrev}',
            'diff_event_ExpOn': f'diff_event_ExpOn_{abbrev}', 'event_resp_rad': f'event_resp_rad_{abbrev}', 
            'exp_dur': f'exp_dur_{abbrev}', 'insp_dur': f'insp_dur_{abbrev}', 
            f'{abbrev}_exp': f'{abbrev}_exp', f'{abbrev}_insp': f'{abbrev}_insp'}
      
        subj_beh_resp = subj_beh_resp.assign(**{col: np.nan for col in columns.values()})

        # Iterate through each row to extract/calculate various RESP features relative to the task event
        for index, row in subj_beh_resp.iterrows():
            event_time = row[event_col] # specify column with timestamps for the task event of interest
            
            # Find expiration onset pre (just before event) and post (just after event)
            expon_pre = max([expon for expon in exp_onsets_s if expon <= event_time], default=np.nan)
            expon_post = min([expon for expon in exp_onsets_s if expon > event_time], default=np.nan)
            subj_beh_resp.at[index, columns['ExpOn_pre']] = expon_pre        # Expiration peak just before task event
            subj_beh_resp.at[index, columns['ExpOn_post']] = expon_post      # Expiration peak just after task event

            if not np.isnan(expon_pre) and not np.isnan(expon_post): 

                # Find timestamp of inspiration onset within expon_pre and expon_post range
                insp_on = next((inspon for inspon in insp_onsets_s if expon_pre < inspon < expon_post), np.nan)
                subj_beh_resp.at[index, columns['InspOn']] = insp_on if insp_on else np.nan
                
                # Calculate respiratory cycle duration around event onset (in sec)
                resp_cycle_dur = expon_post - expon_pre    # respiratory cycle duration as distance between expiration peaks pre- and post-event
                subj_beh_resp.at[index, columns['resp_cycle_dur']] = round(resp_cycle_dur,3)

                # Calculate instantaneous respiratory rate (RR_resp) per min (in breaths per minute - bpm) and per sec (in Hz) based on respiratory cycle length
                rr_resp_bpm = 60 / resp_cycle_dur   # instantaneous RR_resp in bpm
                rr_resp_hz = 1 / resp_cycle_dur     # instantaneous RR_resp in Hz 
                subj_beh_resp.at[index, columns['RR_resp_1perMin']] = rr_resp_bpm
                subj_beh_resp.at[index, columns['RR_resp_1perSec']] = rr_resp_hz

                # Calculate time difference (in sec) from task event to the preceding Exp onset (expon_pre)
                diff_expon_s = event_time - expon_pre
                subj_beh_resp.at[index, columns['diff_event_ExpOn']] = round(diff_expon_s,3)

                # Extract phase at event time directly from phase vector
                phase_vec = subj_resp_data.get('resp_phase_vec', None)
                if phase_vec is not None and not np.isnan(event_time):
                    event_sample = int(round(event_time * sfreq))
                    if 0 <= event_sample < len(phase_vec):
                        event_resp_rad = phase_vec[event_sample]
                    else:
                        event_resp_rad = np.nan
                else:
                    event_resp_rad = np.nan

                subj_beh_resp.at[index, columns['event_resp_rad']] = event_resp_rad

                # Calculate duration of expiration & inspiration phases
                exp_dur = insp_on - expon_pre
                subj_beh_resp.at[index, columns['exp_dur']] = round(exp_dur,3)
                insp_dur = expon_post - insp_on
                subj_beh_resp.at[index, columns['insp_dur']] = round(insp_dur,3)
                   
                ############## RESP features for binary analysis ##############         
                
                ##### Define in which respiratory phase the event falls #####

                # Check if event onset falls within expiration or inspiration
                subj_beh_resp.at[index, columns[f'{abbrev}_exp']] = int(expon_pre < event_time < insp_on)       # binary of event onset at expiration (descending phase)
                subj_beh_resp.at[index, columns[f'{abbrev}_insp']] = int(insp_on < event_time < expon_post)     # binary of event onset at inspiration (ascending phase)

        beh_resp_dfs.append(subj_beh_resp)

    # Concatenate the dfs for all participants
    beh_resp_df = pd.concat(beh_resp_dfs, ignore_index=True)
    beh_resp_df.rename(columns={'Unnamed: 0': 'subj_index'}, inplace=True)

    ############## (Optional) outlier removal ############## 

    if outlier_remove:
        filtered_dfs = []
        outlier_data = []  # store subjID and outlier count

        for subj in beh_resp_df[participant_col].unique():
            filtered_df = beh_resp_df[beh_resp_df[participant_col] == subj].copy()

            # Calculate mean and std for exp and insp durations
            exp_dur_mean = filtered_df[columns['exp_dur']].mean()
            exp_dur_sd = filtered_df[columns['exp_dur']].std()
            insp_dur_mean = filtered_df[columns['insp_dur']].mean()
            insp_dur_sd = filtered_df[columns['insp_dur']].std()

            # Identify outliers in exp/insp duration that are +/- 4 SD from individual mean 
            is_exp_outlier = (filtered_df[columns['exp_dur']] > exp_dur_mean + 4 * exp_dur_sd) | \
                            (filtered_df[columns['exp_dur']] < exp_dur_mean - 4 * exp_dur_sd)

            is_insp_outlier = (filtered_df[columns['insp_dur']] > insp_dur_mean + 4 * insp_dur_sd) | \
                            (filtered_df[columns['insp_dur']] < insp_dur_mean - 4 * insp_dur_sd)

            # Set NaNs for outlier rows
            any_outlier = is_exp_outlier | is_insp_outlier      # combined outlier mask

            # Count number of excluded trials with outliers in either expiration or inspiration duration
            outliers_n = (any_outlier.sum())
            outliers_percent = round((outliers_n / 180) * 100,2)
            outlier_data.append({'subjID': f'sub-{subj}', 'outliers_n': outliers_n, 'outliers_percent': outliers_percent})

            # Set relevant columns of mask to NaN
            filtered_df.loc[any_outlier, [
                columns['exp_dur'], columns['insp_dur'], 
                columns[f'{abbrev}_exp'], columns[f'{abbrev}_insp'],
                columns['resp_cycle_dur'], columns['RR_resp_1perMin'], columns['RR_resp_1perSec'],
                columns['diff_event_ExpOn'], columns['event_resp_rad']
            ]] = np.nan

            filtered_dfs.append(filtered_df)

        # Concatenate cleaned participant rows
        beh_resp_df = pd.concat(filtered_dfs, ignore_index=True)

        # Store outlier count to DataFrame and print average
        outliers_df = pd.DataFrame(outlier_data)
        outliers_avg = outliers_df['outliers_n'].mean()
        outliers_sd = outliers_df['outliers_n'].std()
        outliers_perc_avg = outliers_df['outliers_percent'].mean()
        utliers_perc_sd = outliers_df['outliers_percent'].std()
        print(f'Average number of outlier trials excluded across participants for {abbrev} conditions: {outliers_avg:.2f} +/- {outliers_sd:.2f}')
        print(f'Average percentage of outlier trials to the total excluded across participants for {abbrev} conditions: {outliers_perc_avg:.2f} +/- {utliers_perc_sd:.2f}')

    
    return beh_resp_df, outliers_df

In [10]:
### Extract RESP features around action (keypress) onset ###

beh_resp_action, beh_resp_action_outliers = extract_resp_features_event(event_col=event_col, abbrev=abbrev, outlier_remove=True)

Average number of outlier trials excluded across participants for act conditions: 3.59 +/- 1.60
Average percentage of outlier trials to the total excluded across participants for act conditions: 1.99 +/- 0.89


In [11]:
### Extract RESP features around tone onset ###

beh_resp_tone, beh_resp_tone_outliers = extract_resp_features_event(event_col=event_col_tone, abbrev=abbrev_tone, outlier_remove=True)

Average number of outlier trials excluded across participants for tone conditions: 3.76 +/- 1.53
Average percentage of outlier trials to the total excluded across participants for tone conditions: 2.09 +/- 0.85


In [10]:
# Save output files as TSV in 'derivatives/resp-phase-analysis' directory

beh_resp_action_dir = os.path.join(respphase_dir, 'task-CardiAgIBTask_beh_resp_action.tsv')
beh_resp_action.to_csv(beh_resp_action_dir, index=False, sep='\t', na_rep="n/a")

beh_resp_tone_dir = os.path.join(respphase_dir, 'task-CardiAgIBTask_beh_resp_tone.tsv')
beh_resp_tone.to_csv(beh_resp_tone_dir, index=False, sep='\t', na_rep="n/a")

### **3b. Descriptives: individual respiratory phases duration**

This section provides descriptives of the individual respiratory phases, including average expiration/inspiration duration for both action and tone trials, ranges of both expiratory and inspiratory intervals, as well as total respiratory cycle duration and average respiratory rate. These are reported in Supplementary Results sections 1.2.3. In addition, the following plots are created:

- Histograms of average expiration & inspiration duration per participant (Figure S3b)

In [11]:
# Create directory for saving plots
project_root = os.path.dirname(wd)
plotting_dir = os.path.join(project_root, 'results', 'datavisualization')
if not os.path.exists(plotting_dir):
    os.makedirs(plotting_dir)

In [12]:
### Descriptives of individual cardiac phases - mean phase duration ###

def summarize_and_merge(action_df, tone_df):
    def summarize(df, subj_col, cols):
        summary = df.groupby(subj_col)[cols].agg(['mean', 'std'])
        summary.columns = ['_'.join(col) for col in summary.columns]
        return summary
    
    action_summary = summarize(action_df, 'subjID', ['exp_dur_act', 'insp_dur_act', 'resp_cycle_dur_act'])
    tone_summary = summarize(tone_df, 'subjID', ['exp_dur_tone', 'insp_dur_tone', 'resp_cycle_dur_tone'])

    return action_summary.join(tone_summary, how='outer')

expinsp_dur_summary = summarize_and_merge(beh_resp_action, beh_resp_tone)

# Expiration
expinsp_dur_summary['exp_dur_mean'] = (expinsp_dur_summary[['exp_dur_act_mean', 'exp_dur_tone_mean']].mean(axis=1))
expinsp_dur_summary['exp_dur_std'] = (expinsp_dur_summary[['exp_dur_act_std', 'exp_dur_tone_std']].mean(axis=1))

# Inspiration
expinsp_dur_summary['insp_dur_mean'] = (expinsp_dur_summary[['insp_dur_act_mean', 'insp_dur_tone_mean']].mean(axis=1))
expinsp_dur_summary['insp_dur_std'] = (expinsp_dur_summary[['insp_dur_act_std', 'insp_dur_tone_std']].mean(axis=1))

# Total respiratory cycle
expinsp_dur_summary['resp_cycle_dur_mean'] = (expinsp_dur_summary[['resp_cycle_dur_act_mean', 'resp_cycle_dur_tone_mean']].mean(axis=1))
expinsp_dur_summary['resp_cycle_dur_std'] = (expinsp_dur_summary[['resp_cycle_dur_act_std', 'resp_cycle_dur_tone_std']].mean(axis=1))
# expinsp_dur_summary['resp_rate_1perMin'] = round((1/expinsp_dur_summary['resp_cycle_dur_mean'])*60,2)
# expinsp_dur_summary['prop_exp2rr'] = expinsp_dur_summary['exp_dur_mean'] / expinsp_dur_summary['resp_rate_1perMin']

print("----- Group-level descriptives: respiratory phase duration around task events -----")
print(f"Total respiratory cycle duration: {round(expinsp_dur_summary['resp_cycle_dur_mean'].mean(),2)} +/- {round(expinsp_dur_summary['resp_cycle_dur_std'].mean(),2)} s")
print(f"Total respiratory cycle range: {round(expinsp_dur_summary['resp_cycle_dur_mean'].min(),2)} - {round(expinsp_dur_summary['resp_cycle_dur_mean'].max(),2)} s")
print(f"Average respiratory rate: {round(1/expinsp_dur_summary['resp_cycle_dur_mean'].mean(),2)} Hz; {round((1/expinsp_dur_summary['resp_cycle_dur_mean'].mean())*60,2)} breaths per minute")

print("\n----- Expiration -----")
print(f"Expiration duration: {round(expinsp_dur_summary['exp_dur_mean'].mean(),2)} +/- {round(expinsp_dur_summary['exp_dur_std'].mean(),2)} s")
print(f"Expiration range: {round(expinsp_dur_summary['exp_dur_mean'].min(),2)} - {round(expinsp_dur_summary['exp_dur_mean'].max(),2)} s")

print("----- Inspiration -----")
print(f"Inspiration duration: {round(expinsp_dur_summary['insp_dur_mean'].mean(),2)} +/- {round(expinsp_dur_summary['insp_dur_std'].mean(),2)} s")
print(f"Inspiration range: {round(expinsp_dur_summary['insp_dur_mean'].min(),2)} - {round(expinsp_dur_summary['insp_dur_mean'].max(),2)} s")

----- Group-level descriptives: respiratory phase duration around task events -----
Total respiratory cycle duration: 4.03 +/- 1.0 s
Total respiratory cycle range: 2.77 - 7.21 s
Average respiratory rate: 0.25 Hz; 14.91 breaths per minute

----- Expiration -----
Expiration duration: 2.66 +/- 0.86 s
Expiration range: 1.75 - 4.98 s
----- Inspiration -----
Inspiration duration: 1.37 +/- 0.36 s
Inspiration range: 1.02 - 2.23 s


In [ ]:
### FIGURE S3b ###
# Plot histograms of average expiration & inspiration duration per participant
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)  
expinsp_dur_summary[['exp_dur_mean', 'insp_dur_mean']].plot.hist(
    density=False, alpha=0.6, bins=30, edgecolor='black', color=['#8E44AD', '#8BBB94'], ax=ax)
ax.set_xlabel('Mean phase duration over participants (s)', fontsize='x-large')
ax.set_ylabel('Number of participants', fontsize='x-large')
ax.set_xlim(0.5, 5.5)
plt.legend(['Expiration', 'Inspiration'])
sns.despine()

# Save plot 
figS3_1 = os.path.join(plotting_dir, "FigureS3b_expinsp_durations.svg")
plt.savefig(figS3_1, format="svg")
plt.show()

## **4. Binary analysis - expiration vs. inspiration ratio of task event [Results 2.3.2]**

This section performs binary analysis of individual respiratory phases (i.e., expiration vs. inspiration) with respect to a task event of interest (user-defined). To take into account between-subject differences in respiratory rate (RR_resp) and respiratory phase lengths, the phase-specific proportion of task events, relative to all trials, is normalized by the proportion of the respective respiratory phase in the entire respiratory cycle. Lastly, in group-level analysis, normalized expiration and inspiration ratios are tested using a two-sided paired t-test. If no respiratory effect is present, task events would be randomly distributed across both respiratory phases, and the task event ratio (N events/N trials) should correspond to the respiratory phase ratio (phase length/whole cycle), i.e., ratio value = 1. If there is a respiratory effect, the phase-specific ratio value should be > 1, indicating over-proportional accumulation of task events in the respective phase. 

In detail, this section includes: 

- **4a. Define normalized ratio for both phases (expiration, inspiration)**: the sum of task events (e.g., keypress or stimulus onset) per each phase (as ratio of all trials that include the task event) is normalized to the proportion of the subject-specific phase length in the total respiratory cycle. The expiration or inspiration ratio is thus derived from: (N events per phase / total N trials) / (individual phase length/ individual mean resp cycle length). 
- **4b. Run group-level binary analysis on expiration vs. inspiration ratio**: at the group level, the normalized expiration and inspiration ratios for the task event are tested against each other using a two-sided paired t-test. This is reported in Results section 2.3.2. 
- **4c. Scatter plot for expiration vs. inspiration ratio**: create a scatterplot showing expiration vs. inspiration ratio for each participant, with dashed lines tracing the expected phase-specific ratio of task event onsets if uniformly distributed (i.e., ratio = 1). Instead, purple markers indicate participants with an over-proportional accumulation of task events in expiration (expiration ratio >1, inspiration <1), green markers indicate participants with an over-proportional accumulation of task events in inspiration (inspiration ratio >1, expiration <1), whilst grey markers indicate participants with no phase preference. 
- **4d. Condition-wise analysis of expiration vs. inspiration ratio**: at the group level, separately per condition of the intentional binding task, the normalized expiration and inspiration ratios for either action or tone trials are tested against each other using a a two-sided paired t-test. This is also reported in Results section 2.3.2. 

In [12]:
############## 4a. Define normalized ratio for both phases (expiration, inspiration) ##############

# Define a custom function for computing the normalized expiration and inspiration ratios: 
# (N events per phase / total N trials) / (individual mean phase length/ individual mean R-R length)

def analyse_expinsp_ratio(beh_resp_df, abbrev, participant_col=column_map['participant']):
    """
    Compute the normalized expiration and inspiration ratios for respiratory phase binary analysis. 

    Parameters:
        -----------
        beh_resp_df : DataFrame
            DataFrame generated in the previous steps using the extract_resp_features_event() function, containing
            preprocessed behavioral data and relevant RESP features around user-defined task-event. 
        abbrev : str
            Abbreviated name for the event type (e.g., 'act'), used as suffix for all the derived columns. 
            Should be the same as the one used in the extract_resp_features_event() function to generate beh_resp_df. 
        participant_col : str
            Column name storing the participant IDs.   

    Returns:
        --------
        DataFrame:
            DataFrame with the wide-format participant data used for respiratory phase binary analysis, 
            incl. count of task events per each phase (exp, insp) normalized by phase proportion. 

    """
    
    # Specify the task events columns per each phase 
    exp_bin = f'{abbrev}_exp' 
    insp_bin = f'{abbrev}_insp'
    exp_dur_ep = f'exp_dur_{abbrev}'
    insp_dur = f'insp_dur_{abbrev}'
    
    # Summarize the count of task events per each phase (exp, insp), based on selected method
    event_phase_ratio_df = beh_resp_df.pivot_table(index=participant_col, values=[exp_bin, insp_bin],
                                                                                 aggfunc=lambda x: (x == 1).sum())
    

    # Compute total number of trials that include task events across respiratory phases
    # NB: if N trials is fixed for all conditions (with no exclusions), it should equal the expected N trials from your design
    event_phase_ratio_df[f'{abbrev}_tot_ntrials'] = (event_phase_ratio_df[exp_bin] + event_phase_ratio_df[insp_bin])
    
    # Compute the participant-specific average of respiratory cycle, plus exp/insp durations
    event_phase_dur = beh_resp_df.pivot_table(index=participant_col, 
                                            values=[f'resp_cycle_dur_{abbrev}', exp_dur_ep, insp_dur])
    event_phase_ratio_df = event_phase_ratio_df.join(event_phase_dur) # join the summary tables

    # Compute normalized expiration and inspiration ratio
    # (N events per phase / total N trials) / (individual mean phase length/ individual mean R-R length)
    event_phase_ratio_df[f'{abbrev}_exp_ratio'] = (
        (event_phase_ratio_df[exp_bin] / event_phase_ratio_df[f'{abbrev}_tot_ntrials']) / 
        (event_phase_ratio_df[exp_dur_ep] / event_phase_ratio_df[f'resp_cycle_dur_{abbrev}']))

    event_phase_ratio_df[f'{abbrev}_insp_ratio'] = (
        (event_phase_ratio_df[insp_bin] / event_phase_ratio_df[f'{abbrev}_tot_ntrials']) / 
        (event_phase_ratio_df[insp_dur] / event_phase_ratio_df[f'resp_cycle_dur_{abbrev}']))
    
    # Compute proportion of respiratory phase intervals relative to total respiratory cycle
    # (individual mean phase length/ individual mean R-R length)
    event_phase_ratio_df[f'{abbrev}_exp_prop2dur'] = event_phase_ratio_df[exp_dur_ep] / event_phase_ratio_df[f'resp_cycle_dur_{abbrev}']
    event_phase_ratio_df[f'{abbrev}_insp_prop2dur'] = event_phase_ratio_df[insp_dur] / event_phase_ratio_df[f'resp_cycle_dur_{abbrev}']

    # Re-organize column order and reset participant ID index 
    event_phase_ratio_df = event_phase_ratio_df.reset_index()
    col_order = [participant_col, exp_bin, insp_bin, f'{abbrev}_tot_ntrials', 
                 f'resp_cycle_dur_{abbrev}', exp_dur_ep, insp_dur, f'{abbrev}_exp_ratio', 
                 f'{abbrev}_insp_ratio', f'{abbrev}_exp_prop2dur', f'{abbrev}_insp_prop2dur']
    event_phase_ratio_df = event_phase_ratio_df[col_order]

    return event_phase_ratio_df

In [13]:
### Calculate normalized expiration & inspiration ratios for action onset ###

act_expinsp_ratio = analyse_expinsp_ratio(beh_resp_df=beh_resp_action, abbrev=abbrev)

In [14]:
### Calculate normalized expiration & inspiration ratios for tone onset ###

tone_expinsp_ratio = analyse_expinsp_ratio(beh_resp_df=beh_resp_tone, abbrev=abbrev_tone)

### **4b. Run group-level binary analysis on expiration vs. inspiration ratio**

In [15]:
############## 4b. Run group-level binary analysis on expiration vs. inspiration ratio ##############

# Group-level binary analysis of action onset: two-sided paired t-test of exp vs. insp ratio
expinsp_ratio_ttest = pg.ttest(act_expinsp_ratio[f'{abbrev}_exp_ratio'], act_expinsp_ratio[f'{abbrev}_insp_ratio'], 
                               paired=True, alternative='two-sided')

# Summary of group-level results: action trials
md("Normalized ratio analysis showed a significantly larger (t({})={}, p={}, Cohen's d={}, 95% CI {}) ratio of action onsets in expiration (M={}, SD={}) as compared to inspiration (M={}, SD={}).".format(
    expinsp_ratio_ttest['dof'].iloc[0], round(expinsp_ratio_ttest['T'].iloc[0],3), 
    round(expinsp_ratio_ttest['p_val'].iloc[0],5), round(expinsp_ratio_ttest['cohen_d'].iloc[0],3), expinsp_ratio_ttest['CI95'].iloc[0], 
    round(act_expinsp_ratio[f'{abbrev}_exp_ratio'].mean(),2), round(act_expinsp_ratio[f'{abbrev}_exp_ratio'].std(),2), 
    round(act_expinsp_ratio[f'{abbrev}_insp_ratio'].mean(),2), round(act_expinsp_ratio[f'{abbrev}_insp_ratio'].std(),2)
))

Normalized ratio analysis showed a significantly larger (t(40)=3.665, p=0.00072, Cohen's d=1.079, 95% CI [0.08 0.26]) ratio of action onsets in expiration (M=1.06, SD=0.1) as compared to inspiration (M=0.89, SD=0.2).

In [16]:
############## 4b. Run group-level binary analysis on expiration vs. inspiration ratio ##############

# Group-level binary analysis of tone onset: two-sided paired t-test of exp vs. insp ratio
tone_expinsp_ratio_ttest = pg.ttest(tone_expinsp_ratio[f'{abbrev_tone}_exp_ratio'], tone_expinsp_ratio[f'{abbrev_tone}_insp_ratio'], 
                                    paired=True, alternative='two-sided')

# Summary of group-level results: tone trials
md("Normalized ratio analysis showed a significantly larger (t({})={}, p={}, Cohen's d={}, 95% CI {}) ratio of tone onsets in expiration (M={}, SD={}) as compared to inspiration (M={}, SD={}).".format(
    tone_expinsp_ratio_ttest['dof'].iloc[0], round(tone_expinsp_ratio_ttest['T'].iloc[0],3), 
    round(tone_expinsp_ratio_ttest['p_val'].iloc[0],5), round(tone_expinsp_ratio_ttest['cohen_d'].iloc[0],3), tone_expinsp_ratio_ttest['CI95'].iloc[0],
    round(tone_expinsp_ratio[f'{abbrev_tone}_exp_ratio'].mean(),2), round(tone_expinsp_ratio[f'{abbrev_tone}_exp_ratio'].std(),2), 
    round(tone_expinsp_ratio[f'{abbrev_tone}_insp_ratio'].mean(),2), round(tone_expinsp_ratio[f'{abbrev_tone}_insp_ratio'].std(),2)
))

Normalized ratio analysis showed a significantly larger (t(40)=3.466, p=0.00127, Cohen's d=1.016, 95% CI [0.06 0.24]) ratio of tone onsets in expiration (M=1.05, SD=0.09) as compared to inspiration (M=0.9, SD=0.19).

### **4c. Scatter plot for expiration vs. inspiration ratio**

In [20]:
############## 4c. Scatter plot for expiration vs. inspiration ratio ##############

def plot_expinsp_ratio(event_phase_ratio_df, abbrev, plot_fname, 
                       participant_col=column_map['participant'], xy_lim=[0.4,1.6]):
    """
    Create scatterplot showing expiration vs. inspiration ratio for each participant. 
    Dashed lines indicate expected phase-specific ratio of task event if uniformly distributed (i.e., ratio = 1). 
    Purple markers indicate expiration preference (expiration ratio >1, inspiration <1), green markers indicate 
    inspiration preference (inspiration ratio >1, expiration <1), grey markers indicate no phase preference. 
    """

    # Melt the event_phase_ratio_df to long format
    event_phase_ratio_df_long = event_phase_ratio_df.melt(id_vars=participant_col,
                                                          value_vars=[f'{abbrev}_exp_ratio', f'{abbrev}_insp_ratio'],
                                                          var_name='phase', value_name=f'{abbrev}_relratio')

    # Aggregate summary of mean, SD and count per respiratory phase ratio
    event_phase_ratio_agg = event_phase_ratio_df_long.groupby('phase').agg(
        mean=(f'{abbrev}_relratio', 'mean'),
        sd=(f'{abbrev}_relratio', 'std'),
        count=(f'{abbrev}_relratio', 'count')).reset_index()

    # Compute standard error and confidence intervals
    event_phase_ratio_agg['se'] = event_phase_ratio_agg['sd'] / np.sqrt(event_phase_ratio_agg['count'])
    event_phase_ratio_agg['se_up'] = event_phase_ratio_agg['mean'] + event_phase_ratio_agg['se']
    event_phase_ratio_agg['se_down'] = event_phase_ratio_agg['mean'] - event_phase_ratio_agg['se']

    # Compute differences between task event rate (i.e., N events per phase to N trials) and phase rate (i.e., phase length to respiratory cycle duration)
    event_phase_ratio_df[f'{abbrev}_diff_ratio2exp'] = ((event_phase_ratio_df[f'{abbrev}_exp'] / event_phase_ratio_df[f'{abbrev}_tot_ntrials']) - 
                                                       event_phase_ratio_df[f'{abbrev}_exp_prop2dur']) # event rate exp - proportion exp
    event_phase_ratio_df[f'{abbrev}_check_ratio2exp'] = np.sign(event_phase_ratio_df[f'{abbrev}_diff_ratio2exp']) # check whether event rate (1) or exp rate (-1) is higher

    event_phase_ratio_df[f'{abbrev}_diff_ratio2insp'] = ((event_phase_ratio_df[f'{abbrev}_insp'] / event_phase_ratio_df[f'{abbrev}_tot_ntrials']) - 
                                                        event_phase_ratio_df[f'{abbrev}_insp_prop2dur']) # event rate insp - proportion insp 
    event_phase_ratio_df[f'{abbrev}_check_ratio2insp'] = np.sign(event_phase_ratio_df[f'{abbrev}_diff_ratio2insp']) # check whether event rate (1) or insp rate (-1) is higher

    # Define conditions for coloring
    conditions = [
        (event_phase_ratio_df[f'{abbrev}_check_ratio2insp'] > 0) & (event_phase_ratio_df[f'{abbrev}_check_ratio2exp'] < 0),  # Q1: left up, overprop insp ratio
        (event_phase_ratio_df[f'{abbrev}_check_ratio2insp'] > 0) & (event_phase_ratio_df[f'{abbrev}_check_ratio2exp'] > 0),  # Q2: right up, non-defined
        (event_phase_ratio_df[f'{abbrev}_check_ratio2insp'] < 0) & (event_phase_ratio_df[f'{abbrev}_check_ratio2exp'] > 0)   # Q3: right down, overprop exp ratio
    ]                                                                                                                        # Q4: left down, non-defined (same as Q2)

    values = [1, 2, 3]  # Define corresponding values for coloring Q1, Q2/Q4, Q3
    event_phase_ratio_df['color'] = np.select(conditions, values, default=2)  # default color is 2 (non-defined)
    event_phase_ratio_df['color'] = event_phase_ratio_df['color'].astype('category') # convert to categorical

    # Define color palette (green: insp, purple: exp, grey: non-defined)
    color_palette = {1: '#8BBB94', 2: '#969696', 3: '#8E44AD'}
    markers_palette = {1: "o", 2: "^", 3: "s"}

    # Create scatter plot
    plt.figure(figsize=(5, 5), dpi=300)
    ax = sns.scatterplot(data=event_phase_ratio_df,
                        x=f'{abbrev}_exp_ratio', y=f'{abbrev}_insp_ratio', # x-axis expiration ratio, y-axis inspiration ratio
                        hue='color', style='color', palette=color_palette,
                        markers=markers_palette, legend=False, s=50)

    # Dashed reference lines at x=1 and y=1 (expected normal distribution of phase-specific ratio)
    plt.axhline(1, linestyle="dashed", color="black", linewidth=0.8)
    plt.axvline(1, linestyle="dashed", color="black", linewidth=0.8)
    
    # Plot mean point of exp/insp ratio with error bars
    plt.errorbar(x=event_phase_ratio_agg["mean"][0],    # exp ratio mean
                y=event_phase_ratio_agg["mean"][1],     # insp ratio mean
                xerr=event_phase_ratio_agg["se"][0],
                yerr=event_phase_ratio_agg["se"][1],
                fmt="o", color="black", markersize=6, capsize=4)

    plt.xlim(xy_lim[0], xy_lim[1])
    plt.ylim(xy_lim[0], xy_lim[1])
    plt.xlabel("Expiration ratio", fontsize='large')
    plt.ylabel("Inspiration ratio", fontsize='large')
    sns.despine()

    #Save plot as SVG
    fig = os.path.join(plotting_dir, plot_fname)
    plt.savefig(fig, format="svg", bbox_inches='tight')
    plt.show()

    return ax, event_phase_ratio_df

In [ ]:
### FIGURE 4b ###
# Plot expiration vs. inspiration ratio of action onsets across all conditions
fig, event_phase_ratio_df = plot_expinsp_ratio(act_expinsp_ratio, 'act', 'Figure4b_expinsp_ratio_action.svg')

In [ ]:
# Plot expiration vs. inspiration ratio of tone onsets across all conditions
fig2, event_phase_ratio_df_tone = plot_expinsp_ratio(tone_expinsp_ratio, 'tone', 'SupplFigure_expinsp_ratio_tone.svg')

### **4d. Condition-wise analysis of expiration vs. inspiration ratio**

In [23]:
### Calculate normalized expiration & inspiration ratios for action onset - BasA condition ###

# Sub-select only BasA condition
beh_resp_action_BasA = beh_resp_action[beh_resp_action[column_map['condition']].isin(['BasA'])]

# Compute the normalized ratios for the BasA condition only
act_expinsp_ratio_BasA = analyse_expinsp_ratio(beh_resp_df=beh_resp_action_BasA, abbrev=abbrev)

# Group-level binary analysis of action onset: two-sided paired t-test of exp vs. ins ratio
expinsp_ratio_ttest_BasA = pg.ttest(act_expinsp_ratio_BasA[f'{abbrev}_exp_ratio'], act_expinsp_ratio_BasA[f'{abbrev}_insp_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on expiratory-inspiratory ratios of action onset in BasA only:\n\n", expinsp_ratio_ttest_BasA)

# Plot expiratory vs. inspiratory ratio of action onsets of BasA condition only
#fig_BasA, act_phase_ratio_df_BasA = plot_expinsp_ratio(act_expinsp_ratio_BasA, 'act', 'Figure7_expinsp_ratio_action_BasA.svg')

t-test on expiratory-inspiratory ratios of action onset in BasA only:

                T  dof alternative     p_val         CI95   cohen_d  power  \
T_test  3.754947   40   two-sided  0.000552  [0.1, 0.33]  1.113656    1.0   

          BF10  
T_test  51.565  


In [24]:
### Calculate normalized expiration & inspiration ratios for action onset - OpA condition ###

# Sub-select only OpA condition
beh_resp_action_OpA = beh_resp_action[beh_resp_action[column_map['condition']].isin(['OpA'])]

# Compute the normalized ratios for the OpA condition only
act_expinsp_ratio_OpA = analyse_expinsp_ratio(beh_resp_df=beh_resp_action_OpA, abbrev=abbrev)

# Group-level binary analysis of action onset: two-sided paired t-test of exp vs. ins ratio
expinsp_ratio_ttest_OpA = pg.ttest(act_expinsp_ratio_OpA[f'{abbrev}_exp_ratio'], act_expinsp_ratio_OpA[f'{abbrev}_insp_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on expiratory-inspiratory ratios of action onset in OpA only:\n\n", expinsp_ratio_ttest_OpA)

# Plot expiratory vs. inspiratory ratio of action onsets of OpA condition only
#fig_OpA, act_phase_ratio_df_OpA = plot_expinsp_ratio(act_expinsp_ratio_OpA, 'act', 'Figure7_expinsp_ratio_action_OpA.svg')

t-test on expiratory-inspiratory ratios of action onset in OpA only:

                T  dof alternative     p_val          CI95   cohen_d    power  \
T_test  2.646389   40   two-sided  0.011577  [0.04, 0.28]  0.786463  0.99842   

         BF10  
T_test  3.567  


In [25]:
### Calculate normalized expiration & inspiration ratios for action onset - OpT condition ###

# Sub-select only OpT condition
beh_resp_action_OpT = beh_resp_action[beh_resp_action[column_map['condition']].isin(['OpT'])]

# Compute the normalized ratios for the OpT condition only
act_expinsp_ratio_OpT = analyse_expinsp_ratio(beh_resp_df=beh_resp_action_OpT, abbrev=abbrev)

# Group-level binary analysis of action onset: two-sided paired t-test of exp vs. ins ratio
expinsp_ratio_ttest_OpT = pg.ttest(act_expinsp_ratio_OpT[f'{abbrev}_exp_ratio'], act_expinsp_ratio_OpT[f'{abbrev}_insp_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on expiratory-inspiratory ratios of action onset in OpT only:\n\n", expinsp_ratio_ttest_OpT)

# Plot expiratory vs. inspiratory ratio of action onsets of OpT condition only
#fig_OpT, act_phase_ratio_df_OpT = plot_expinsp_ratio(act_expinsp_ratio_OpT, 'act', 'Figure7_expinsp_ratio_action_OpT.svg')

t-test on expiratory-inspiratory ratios of action onset in OpT only:

                T  dof alternative     p_val          CI95   cohen_d     power  \
T_test  2.354657   40   two-sided  0.023535  [0.02, 0.27]  0.697264  0.991694   

         BF10  
T_test  1.962  


In [26]:
### Calculate normalized expiration & inspiration ratios for tone onset - BasT condition ###

# Sub-select only BasT conditions
beh_resp_tone_BasT = beh_resp_tone[beh_resp_tone[column_map['condition']].isin(['BasT'])]

# Compute the normalized ratios for the BasT condition only
tone_expinsp_ratio_BasT = analyse_expinsp_ratio(beh_resp_df=beh_resp_tone_BasT, abbrev=abbrev_tone)

# Group-level binary analysis of action onset: two-sided paired t-test of exp vs. ins ratio
expinsp_ratio_tone_ttest_BasT = pg.ttest(tone_expinsp_ratio_BasT[f'{abbrev_tone}_exp_ratio'], tone_expinsp_ratio_BasT[f'{abbrev_tone}_insp_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on expiratory-inspiratory ratios of tone onset in BasT only:\n\n", expinsp_ratio_tone_ttest_BasT)

# Plot expiratory vs. inspiratory ratio of action onsets of BasT condition only
#fig_tone_BasT, event_phase_ratio_df_BasT = plot_expinsp_ratio(tone_expinsp_ratio_BasT, 'tone', 'Figure7_expinsp_ratio_tone_BasT.svg')

t-test on expiratory-inspiratory ratios of tone onset in BasT only:

                T  dof alternative     p_val           CI95   cohen_d  \
T_test  0.537183   40   two-sided  0.594117  [-0.08, 0.13]  0.157951   

           power   BF10  
T_test  0.166947  0.193  


In [27]:
### Calculate normalized expiration & inspiration ratios for tone onset - OpT condition ###

# Sub-select only OpT conditions
beh_resp_tone_OpT = beh_resp_tone[beh_resp_tone[column_map['condition']].isin(['OpT'])]

# Compute the normalized ratios for the OpT condition only
tone_expinsp_ratio_OpT = analyse_expinsp_ratio(beh_resp_df=beh_resp_tone_OpT, abbrev=abbrev_tone)

# Group-level binary analysis of action onset: two-sided paired t-test of exp vs. ins ratio
expinsp_ratio_tone_ttest_OpT = pg.ttest(tone_expinsp_ratio_OpT[f'{abbrev_tone}_exp_ratio'], tone_expinsp_ratio_OpT[f'{abbrev_tone}_insp_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on expiratory-inspiratory ratios of tone onset in OpT only:\n\n", expinsp_ratio_tone_ttest_OpT)

# Plot expiratory vs. inspiratory ratio of action onsets of OpT condition only
#fig_tone_OpT, event_phase_ratio_df_OpT = plot_expinsp_ratio(tone_expinsp_ratio_OpT, 'tone', 'Figure7_expinsp_ratio_tone_OpT.svg')

t-test on expiratory-inspiratory ratios of tone onset in OpT only:

                T  dof alternative     p_val          CI95   cohen_d     power  \
T_test  2.936741   40   two-sided  0.005478  [0.06, 0.33]  0.865509  0.999714   

         BF10  
T_test  6.786  


In [28]:
### Calculate normalized expiration & inspiration ratios for tone onset - OpA condition ###

# Sub-select only OpA conditions
beh_resp_tone_OpA = beh_resp_tone[beh_resp_tone[column_map['condition']].isin(['OpA'])]

# Compute the normalized ratios for the OpA condition only
tone_expinsp_ratio_OpA = analyse_expinsp_ratio(beh_resp_df=beh_resp_tone_OpA, abbrev=abbrev_tone)

# Group-level binary analysis of action onset: two-sided paired t-test of exp vs. ins ratio
expinsp_ratio_tone_ttest_OpA = pg.ttest(tone_expinsp_ratio_OpA[f'{abbrev_tone}_exp_ratio'], tone_expinsp_ratio_OpA[f'{abbrev_tone}_insp_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on expiratory-inspiratory ratios of tone onset in OpA only:\n\n", expinsp_ratio_tone_ttest_OpA)

# Plot expiratory vs. inspiratory ratio of action onsets of OpA condition only
#fig_tone_OpA, event_phase_ratio_df_OpA = plot_expinsp_ratio(tone_expinsp_ratio_OpA, 'tone', 'Figure7_expinsp_ratio_tone_OpA.svg')

t-test on expiratory-inspiratory ratios of tone onset in OpA only:

                T  dof alternative     p_val          CI95   cohen_d  power  \
T_test  3.707033   40   two-sided  0.000635  [0.11, 0.37]  1.100788    1.0   

          BF10  
T_test  45.451  


## **5. Binary analysis: respiratory phase binning of intentional binding [Results 2.3.3 & 2.5.3]**

This section performs binary analysis of a given dependent variable (e.g., action binding, tone binding) acccording to the respiratory phase of the task event onset, i.e., binning into expiration and inspiration. In detail, the following two steps are included:

- **5a. Analyse intentional binding by binning task event onset in expiration or inspiration**: computes respiratory phase differences in the dependent variable (i.e., action binding or tone binding, respectively) are computed. This is reported in Results section 2.3.3 with corresponding Figures 4c, 4d and 4e. 
- **5b. Correlation of intentional binding with individual expiration/inspiration ratios**: performs correlation analysis between the individual normalized expiratory and inspiratory ratios for action/tone trials and the corresponding intentional binding measure (i.e., action binding and tone binding, respectively). This corresponds to Results section 2.5.3, as well as Figures 7c & 7d. 

### **5a. Analyse intentional binding by binning task event onset in expiration or inspiration**

After binning the task events of relevance (i.e., action or tone) into expiration/inspiration, respiratory phase differences in the dependent variable (i.e., action binding or tone binding, respectively) are computed. This is reported in Results section 2.3.3 with corresponding Figures 4c, 4d and 4e. 

In [17]:
############## 5a. Analyse intentional binding by binning task event onset in exp/insp ##############

# Define function to analyze differences in a dependent variable (e.g., action binding) according to 
# whether a task event onset was binned in expiration or inspiration
def analyze_expinsp_binding(beh_resp_df, conditions, abbrev, depend_var,
                            participant_col=column_map['participant'], condition_col=column_map['condition']):
    """
    Analyze the dependent variable from the intentional binding task (action or tone binding),
    according to the task event (i.e., action or tone onset) falling in expiration or inspiration.

    Parameters
    beh_resp_df
        beh_ecg_df : DataFrame
            DataFrame generated in the previous steps using the extract_resp_features_event() function, containing
            preprocessed behavioral data and relevant RESP features around user-defined task-event. 
        conditions : tuple
            Tuple of two condition names to analyze, where first should be baseline condition and 
            second should be operant condition, e.g. ('BasA', 'OpA'). 
        abbrev : str
            Abbreviated name for the event type (e.g., 'act'), used as suffix for all the derived columns. 
            Should be the same as the one used in the extract_resp_features_event() function to generate beh_resp_df.  
        depend_var : str
            Column name including the dependent variable to be analyzed (e.g., 'JE_act')
        participant_col : str, default (column_map['participant'])
            Column name storing the participant IDs.   
        condition_col : str, default (column_map['condition'])
            Column name storing the condition names. 

    Returns
    -----------
    Tuple: (df_means, df_means_long, ttest_res)
        - df_means : Dataframe with means of dependent variable in expiration vs. inspiration (wide format)
        - df_means_long : Long-format dataframe for plotting
        - ttest_res : Paired samples t-test results comparing sys vs. dia


    """
    # Filter data according to exp/insp bins where task event falls
    exp_df = beh_resp_df[beh_resp_df[f'{abbrev}_exp'] == 1]
    insp_df = beh_resp_df[beh_resp_df[f'{abbrev}_insp'] == 1]

    # Define function to calculate means of dependent variable for given conditions in expiration vs. inspiration
    def calculate_means(filtered_data, phase_name):
        filtered_data = filtered_data[filtered_data['condition'].isin(conditions)]  # Filter only selected conditions
        means = (
            filtered_data.groupby([participant_col, condition_col])[f'{depend_var}']
            .mean().unstack()
            .rename(columns={conditions[0]: f'JE_{conditions[0]}_{phase_name}', 
                             conditions[1]: f'JE_{conditions[1]}_{phase_name}'})
        )
        return means
    
    # Compute means for expiration and inspiration
    means_exp = calculate_means(exp_df, 'exp')
    means_insp = calculate_means(insp_df, 'insp')

    # Merge the results and calculate action binding
    df_means = means_exp.join(means_insp, how='outer', lsuffix='_exp', rsuffix='_insp')
    df_means[f'{abbrev}_binding_exp'] = df_means[f'JE_{conditions[1]}_exp'] - df_means[f'JE_{conditions[0]}_exp']
    df_means[f'{abbrev}_binding_insp'] = df_means[f'JE_{conditions[1]}_insp'] - df_means[f'JE_{conditions[0]}_insp']

    # Convert to long format for plotting
    df_means_long = df_means[[f'{abbrev}_binding_exp', f'{abbrev}_binding_insp']].reset_index().melt(
        id_vars=participant_col, var_name='phase', value_name=f'{abbrev}_binding')

    # Paired t-test   
    ttest_res = pg.ttest(df_means[f'{abbrev}_binding_exp'], df_means[f'{abbrev}_binding_insp'], 
                         paired=True, alternative='two-sided')
                                    
    return df_means, df_means_long, ttest_res

In [18]:
# Analyze action binding according to action onset in exp vs. insp
actbind_means, actbind_means_long, actbind_ttest = analyze_expinsp_binding(beh_resp_df=beh_resp_action, conditions=('BasA', 'OpA'), 
                                                                           abbrev=abbrev, depend_var='JE_act')

# Summary of group-level results: action binding according to action onset in exp vs. insp
md("Binary analysis did not show a significant difference (t({})={}, p={}, Cohen's d={}, 95% CI {}) in action binding when actions were initiated at expiration (M={}, SD={}) compared to inspiration (M={}, SD={}).".format(
    actbind_ttest['dof'].iloc[0], round(actbind_ttest['T'].iloc[0],3), 
    round(actbind_ttest['p_val'].iloc[0],3), round(actbind_ttest['cohen_d'].iloc[0],3), actbind_ttest['CI95'].iloc[0], 
    round(actbind_means[f'act_binding_exp'].mean(),2), round(actbind_means[f'act_binding_exp'].std(),2), 
    round(actbind_means[f'act_binding_insp'].mean(),2), round(actbind_means[f'act_binding_insp'].std(),2)
))

Binary analysis did not show a significant difference (t(40)=0.612, p=0.544, Cohen's d=0.088, 95% CI [-6.95 12.99]) in action binding when actions were initiated at expiration (M=34.21, SD=34.74) compared to inspiration (M=31.19, SD=33.84).

In [31]:
# Summary table of respiratory effects on JE in BasA/OpA and action binding

# Define comparisons for t-tests
comparisons_act = {"Judgement Error (ms) in BasA":  ('JE_BasA_exp', 'JE_BasA_insp'),          # Judgment Error in BasA condition
                   "Judgement Error (ms) in OpA":   ('JE_OpA_exp', 'JE_OpA_insp'),            # Judgment Error in OpA condition 
                   "Action Binding (ms)":           ('act_binding_exp', 'act_binding_insp')}  # Action binding (JE_OpA - JE_BasA)

summary_rows = []
for label, (exp_col, insp_col) in comparisons_act.items():
    exp_vals = actbind_means[exp_col]           # extract values for action onset in expiration
    insp_vals = actbind_means[insp_col]         # extract values for action onset in inspiration

    ttest_res = pg.ttest(exp_vals, insp_vals, paired=True, alternative='two-sided')  # paired t-test expiration vs. inspiration
    t_stat = ttest_res['T'].iloc[0]         
    p_val = ttest_res['p_val'].iloc[0]
    
    summary_rows.append({
        "Measure": label,
        "Expiration mean (SD)": f"{exp_vals.mean():.2f} ({exp_vals.std():.2f})",
        "Inspiration mean (SD)": f"{insp_vals.mean():.2f} ({insp_vals.std():.2f})",
        "T-statistic": f"{t_stat:.2f}",
        "p-value": f"{p_val:.4f}"
    })

    if label == 'Action Binding (ms)':
        actbind_respeffect = exp_vals.mean() - insp_vals.mean()  # respiration effect 

# Convert summary table to DataFrame and display
summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))
print(f"\nRespiratory effect (ms) on action binding: {round(actbind_respeffect,3)}")

                     Measure Expiration mean (SD) Inspiration mean (SD) T-statistic p-value
Judgement Error (ms) in BasA        26.70 (67.10)         27.27 (66.94)       -0.14  0.8856
 Judgement Error (ms) in OpA        60.92 (66.88)         58.46 (64.02)        0.70  0.4899
         Action Binding (ms)        34.21 (34.74)         31.19 (33.84)        0.61  0.5439

Respiratory effect (ms) on action binding: 3.019


In [19]:
# Analyze tone binding according to tone onset in exp vs. insp
tonebind_means, tonebind_means_long, tonebind_ttest = analyze_expinsp_binding(beh_resp_df=beh_resp_tone, conditions=('BasT', 'OpT'), 
                                                                              abbrev=abbrev_tone, depend_var='JE_tone')

# Summary of group-level results: tone binding according to tone onset in exp vs. insp
md("Binary analysis showed a significant difference (t({})={}, p={}, Cohen's d={}, 95% CI {}) in tone binding when tones were initiated at expiration (M={}, SD={}) compared to inspiration (M={}, SD={}).".format(
    tonebind_ttest['dof'].iloc[0], round(tonebind_ttest['T'].iloc[0],3), 
    round(tonebind_ttest['p_val'].iloc[0],3), round(tonebind_ttest['cohen_d'].iloc[0],3), tonebind_ttest['CI95'].iloc[0], 
    round(tonebind_means[f'tone_binding_exp'].mean(),2), round(tonebind_means[f'tone_binding_exp'].std(),2), 
    round(tonebind_means[f'tone_binding_insp'].mean(),2), round(tonebind_means[f'tone_binding_insp'].std(),2)
))

Binary analysis showed a significant difference (t(40)=2.374, p=0.022, Cohen's d=0.179, 95% CI [ 2.23 27.81]) in tone binding when tones were initiated at expiration (M=-80.96, SD=79.24) compared to inspiration (M=-95.98, SD=88.59).

In [33]:
# Summary table of respiratory effects on JE in BasT/OpT and tone binding

# Define comparisons for t-tests
comparisons_tone = {"Judgement Error (ms) in BasT":  ('JE_BasT_exp', 'JE_BasT_insp'),            # Judgment Error in BasT condition
                   "Judgement Error (ms) in OpT":    ('JE_OpT_exp', 'JE_OpT_insp'),              # Judgment Error in OpT condition 
                   "Tone Binding (ms)":              ('tone_binding_exp', 'tone_binding_insp')}  # Tone binding (JE_OpT - JE_BasT)

summary_rows = []
for label, (exp_col, insp_col) in comparisons_tone.items():
    exp_vals = tonebind_means[exp_col]           # extract values for tone onset in expiration
    insp_vals = tonebind_means[insp_col]         # extract values for tone onset in inspiration

    ttest_res = pg.ttest(exp_vals, insp_vals, paired=True, alternative='two-sided')  # paired t-test expiration vs. inspiration
    t_stat = ttest_res['T'].iloc[0]         
    p_val = ttest_res['p_val'].iloc[0]
    
    summary_rows.append({
        "Measure": label,
        "Expiration mean (SD)": f"{exp_vals.mean():.2f} ({exp_vals.std():.2f})",
        "Inspiration mean (SD)": f"{insp_vals.mean():.2f} ({insp_vals.std():.2f})",
        "T-statistic": f"{t_stat:.2f}",
        "p-value": f"{p_val:.4f}"
    })

    if label == 'Tone Binding (ms)':
        tonebind_respeffect = exp_vals.mean() - insp_vals.mean()  # respiration effect 

# Convert summary table to DataFrame and display
summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))
print(f"\nRespiratory effect (ms) on tone binding: {round(tonebind_respeffect,3)}")

                     Measure Expiration mean (SD) Inspiration mean (SD) T-statistic p-value
Judgement Error (ms) in BasT        42.53 (65.00)         49.14 (62.02)       -1.64  0.1082
 Judgement Error (ms) in OpT       -38.43 (95.53)        -46.84 (98.99)        2.00  0.0526
           Tone Binding (ms)       -80.96 (79.24)        -95.98 (88.59)        2.37  0.0225

Respiratory effect (ms) on tone binding: 15.02


In [ ]:
############## Summary arrow plot of action & tone binding across respiratory phases ##############

### FIGURE 4e ###

colpalette = ['#8E44AD', '#CDA7DD', '#8BBB94', '#C0D8C5']

# Set figure size and dpi
fig, ax = plt.subplots(figsize=(7.5, 3), dpi=300)

# Draw a horizontal line and labels for time
time_axis = patches.FancyArrowPatch((-50, 0.8), (320, 0.8), color='k', arrowstyle='-|>', mutation_scale=8)
ax.add_patch(time_axis)
ax.text(x=125, y=0.83, s='250 ms', color='k', ha='center', va='center', fontsize=9) # delay label
ax.text(x=310, y=0.76, s='Time', color='k', ha='right', va='center', fontsize=9) # time axis label
int_arrow = patches.FancyArrowPatch((0, 0.77), (250, 0.77), color='grey', arrowstyle='<|-|>', mutation_scale=8, linewidth=0.8)
ax.add_patch(int_arrow)
ax.text(x=125, y=0.74, s='Actual interval', color='grey', ha='center', va='center', fontsize=7) # actual interval label

# Draw line for actual Action time (0 ms)
ax.axvline(x=0, color='k', linestyle='-', linewidth=4, ymin=0.75, ymax=0.85)
ax.text(x=0, y=0.92, s='Action', color='k', ha='center', va='center', fontsize=13)

# Draw line for actual Tone time (250ms)
ax.axvline(x=250, color='k', linestyle='-', linewidth=4, ymin=0.75, ymax=0.85)
ax.text(x=250, y=0.92, s='Tone', color='k', ha='center', va='center', fontsize=13)

# Draw perceived Baseline/Operant times for Expiration 
ax.axvline(x=actbind_means['JE_BasA_exp'].mean(), color=colpalette[1], linestyle='-', linewidth=4, ymin=0.55, ymax=0.65)
ax.axvline(x=(250 + tonebind_means['JE_BasT_exp'].mean()), color=colpalette[1], linestyle='-', linewidth=4, ymin=0.55, ymax=0.65)
ax.axvline(x=actbind_means['JE_OpA_exp'].mean(), color=colpalette[0], linestyle='-', linewidth=4, ymin=0.55, ymax=0.65)
ax.axvline(x=(250 + tonebind_means['JE_OpT_exp'].mean()), color=colpalette[0], linestyle='-', linewidth=4, ymin=0.55, ymax=0.65)
ax.text(x=-80, y=0.6, s='Expiration', color='k', ha='left', va='center', fontsize=11)

# Draw perceived Baseline/Operant times for Inspiration
ax.axvline(x=actbind_means['JE_BasA_insp'].mean(), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.35, ymax=0.45)
ax.axvline(x=(250 + tonebind_means['JE_BasT_insp'].mean()), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.35, ymax=0.45)
ax.axvline(x=actbind_means['JE_OpA_insp'].mean(), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.35, ymax=0.45)
ax.axvline(x=(250 + tonebind_means['JE_OpT_insp'].mean()), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.35, ymax=0.45)
ax.text(x=-80, y=0.4, s='Inspiration', color='k', ha='left', va='center', fontsize=11)

# Draw arrow and add text for Action Binding across Expiration-Inspiration
ax.arrow(x=(actbind_means['JE_BasA_exp'].mean()+1), y=0.6, dx=(actbind_means['JE_OpA_exp'].mean() - actbind_means['JE_BasA_exp'].mean())*0.87, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(actbind_means['JE_BasA_exp'].mean() + (actbind_means['JE_OpA_exp'].mean() - actbind_means['JE_BasA_exp'].mean())/2)*0.88, y=0.56, 
        s=f"{actbind_means['act_binding_exp'].mean():.2f} ({actbind_means['act_binding_exp'].std():.2f}) ms", color=colpalette[0], ha='center', va='center', fontsize=5)
ax.arrow(x=(actbind_means['JE_BasA_insp'].mean()+1), y=0.4, dx=(actbind_means['JE_OpA_insp'].mean() - actbind_means['JE_BasA_insp'].mean())*0.865, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(actbind_means['JE_BasA_insp'].mean() + (actbind_means['JE_OpA_insp'].mean() - actbind_means['JE_BasA_insp'].mean())/2)*0.86, y=0.36, 
        s=f"{actbind_means['act_binding_insp'].mean():.2f} ({actbind_means['act_binding_insp'].std():.2f}) ms", color=colpalette[2], ha='center', va='center', fontsize=5)

# Draw arrow and add text for Tone Binding across Expiration-Inspiration
ax.arrow(x=(250 + tonebind_means['JE_BasT_exp'].mean()), y=0.6, dx=(tonebind_means['JE_OpT_exp'].mean() - tonebind_means['JE_BasT_exp'].mean())*0.94, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + tonebind_means['JE_BasT_exp'].mean())+((tonebind_means['JE_OpT_exp'].mean() - tonebind_means['JE_BasT_exp'].mean())/2)*0.95), y=0.56, 
         s=f"{tonebind_means['tone_binding_exp'].mean():.2f} ({tonebind_means['tone_binding_exp'].std():.2f}) ms", color=colpalette[0], ha='center', va='center', fontsize=5)
ax.arrow(x=(250 + tonebind_means['JE_BasT_insp'].mean()), y=0.4, dx=(tonebind_means['JE_OpT_insp'].mean() - tonebind_means['JE_BasT_insp'].mean())*0.95, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + tonebind_means['JE_BasT_insp'].mean())+((tonebind_means['JE_OpT_insp'].mean() - tonebind_means['JE_BasT_insp'].mean())/2)*0.95), y=0.36,  
         s=f"{tonebind_means['tone_binding_insp'].mean():.2f} ({tonebind_means['tone_binding_insp'].std():.2f}) ms", color=colpalette[2], ha='center', va='center', fontsize=5)

# Set limits and remove axes
ax.set_xlim(-100, 350)
ax.set_ylim(0, 1)
ax.axis('off')
plt.tight_layout()

# Save plot 
fig8 = os.path.join(plotting_dir, "Figure4e_respiratory_arrow_summary.svg")
plt.savefig(fig8, format="svg")
plt.show()

In [35]:
##### Save the respiratory phase binning datasets for action binding & tone binding #####

actbind_dir = os.path.join(respphase_dir, 'task-CardiAgIBTask_resp_actbinding_avg.tsv')
actbind_means_long.to_csv(actbind_dir, index=False, sep='\t', na_rep="n/a")

tonebind_dir = os.path.join(respphase_dir, 'task-CardiAgIBTask_resp_tonebinding_avg.tsv')
tonebind_means_long.to_csv(tonebind_dir, index=False, sep='\t', na_rep="n/a")

### **5b. Correlation of intentional binding with individual expiration/inspiration ratios**

This section performs correlation analysis between the individual normalized expiratory and inspiratory ratios for action/tone trials and the corresponding intentional binding measure (i.e., action binding and tone binding, respectively). This corresponds to Results section 2.5.3, as well as Figures 7c & 7d. 

In [20]:
############## 5b. Correlation of intentional binding with individual exp/insp ratios ##############

# Sub-select only BasA and OpA conditions
beh_resp_action_BasAOpA = beh_resp_action[beh_resp_action[column_map['condition']].isin(['BasA', 'OpA'])]

# Run expiration/inspiration ratio analysis on BasA/OpA conditions
act_expinsp_ratio_BasAOpA = analyse_expinsp_ratio(beh_resp_df=beh_resp_action_BasAOpA, abbrev=abbrev)

# Summarize mJE (sd) for action and tone trials by condition and subject
IBmeasures_BasAOpA = beh_resp_action.groupby(["subjID", "condition"])['JE_act'].mean().unstack()
IBmeasures_BasAOpA = IBmeasures_BasAOpA.filter(items=['BasA', 'OpA']).rename(columns={'OpA': 'JE_OpA', 'BasA': 'JE_BasA'})
IBmeasures_BasAOpA['act_binding'] = IBmeasures_BasAOpA['JE_OpA'] - IBmeasures_BasAOpA['JE_BasA']

# Merge together expiration/inspiration ratios, and summary of JE_act per condition and subject
act_expinsp_ratio_actbind_BasAOpA = act_expinsp_ratio_BasAOpA.join(IBmeasures_BasAOpA, how='left', on='subjID')
act_expinsp_ratio_actbind_BasAOpA = act_expinsp_ratio_actbind_BasAOpA.join(actbind_means, how='left', on='subjID')

In [21]:
### Correlation of expiration ratio and action binding ###
exp_actbind_corr = pg.corr(x=act_expinsp_ratio_actbind_BasAOpA['act_exp_ratio'], y=act_expinsp_ratio_actbind_BasAOpA['act_binding'])

md("No significant correlation between individual expiration ratios and action binding (r({})={}, p={}, 95% CI {}).".format(
    (exp_actbind_corr['n'].iloc[0]-2), round(exp_actbind_corr['r'].iloc[0],3), 
    round(exp_actbind_corr['p_val'].iloc[0],3), exp_actbind_corr['CI95'].iloc[0]
))

No significant correlation between individual expiration ratios and action binding (r(39)=0.007, p=0.965, 95% CI [-0.3   0.31]).

In [22]:
### Correlation of inspiration ratio and action binding ###
insp_actbind_corr = pg.corr(x=act_expinsp_ratio_actbind_BasAOpA['act_insp_ratio'], y=act_expinsp_ratio_actbind_BasAOpA['act_binding'])

md("No significant correlation between individual inspiration ratios and action binding (r({})={}, p={}, 95% CI {}).".format(
    (insp_actbind_corr['n'].iloc[0]-2), round(insp_actbind_corr['r'].iloc[0],3), 
    round(insp_actbind_corr['p_val'].iloc[0],3), insp_actbind_corr['CI95'].iloc[0]
))

No significant correlation between individual inspiration ratios and action binding (r(39)=0.038, p=0.814, 95% CI [-0.27  0.34]).

In [ ]:
### FIGURE 7c ###
# Correlation plot of action binding and individual exp/insp ratios

plt.figure(figsize=(6, 6), dpi=300)

# Add horizontal line for group-level mean action binding effect
actbind_group_avg = 32.87
plt.axhline(actbind_group_avg, linestyle="dashed", color="k", linewidth=0.8, alpha=0.3)

# Scatter and regression line for act_exp_ratio
sns.regplot(data=act_expinsp_ratio_actbind_BasAOpA, x='act_exp_ratio', y='act_binding', marker= 's', 
            scatter_kws={'color': '#8E44AD', 'alpha': 0.7}, line_kws={'color': '#8E44AD'}, label="Expiration Ratio")

# Scatter and regression line for act_insp_ratio
sns.regplot(data=act_expinsp_ratio_actbind_BasAOpA, x='act_insp_ratio', y='act_binding',
            scatter_kws={'color': '#8BBB94', 'alpha': 0.7}, line_kws={'color':'#8BBB94'}, label="Inspiration Ratio")

plt.ylabel("Action Binding: OpA - BasA (ms)", fontsize='x-large')
plt.xlabel("Respiratory phase ratios", fontsize='x-large')
plt.xlim(0.3, 1.5)
plt.legend(loc='upper right')
sns.despine()

# Save plot 
fig10_3 = os.path.join(plotting_dir, "Figure7c_expinsp_ratio_actbind_correlation.svg")
plt.savefig(fig10_3, format="svg")
plt.show()

In [23]:
############## 7b. Correlation of dependent variable (i.e., tone binding) with individual exp/insp ratios ##############

# Sub-select only BasT and OpT conditions
beh_resp_tone_BasTOpT = beh_resp_tone[beh_resp_tone[column_map['condition']].isin(['BasT', 'OpT'])]

# Run systolic/diastolic ratio analysis on BasT/OpT conditions
tone_expinsp_ratio_BasTOpT = analyse_expinsp_ratio(beh_resp_df=beh_resp_tone_BasTOpT, abbrev=abbrev_tone)

# Summarize mJE (sd) for action and tone trials by condition and subject
IBmeasures_BasTOpT = beh_resp_tone.groupby(["subjID", "condition"])['JE_tone'].mean().unstack()
IBmeasures_BasTOpT = IBmeasures_BasTOpT.filter(items=['BasT', 'OpT']).rename(columns={'OpT': 'JE_OpT', 'BasT': 'JE_BasT'})
IBmeasures_BasTOpT['tone_binding'] = IBmeasures_BasTOpT['JE_OpT'] - IBmeasures_BasTOpT['JE_BasT']

# Merge together expiration/inspiration ratios, and summary of JE_act per condition and subject
tone_expinsp_ratio_tonebind_BasTOpT = tone_expinsp_ratio_BasTOpT.join(IBmeasures_BasTOpT, how='left', on='subjID')
tone_expinsp_ratio_tonebind_BasTOpT = tone_expinsp_ratio_tonebind_BasTOpT.join(tonebind_means, how='left', on='subjID')

In [24]:
# Correlation of expiration ratio and tone binding
exp_tonebind_corr = pg.corr(x=tone_expinsp_ratio_tonebind_BasTOpT['tone_exp_ratio'], y=tone_expinsp_ratio_tonebind_BasTOpT['tone_binding'])

md("No significant correlation between individual expiration ratios and tone binding (r({})={}, p={}, 95% CI {}).".format(
    (exp_tonebind_corr['n'].iloc[0]-2), round(exp_tonebind_corr['r'].iloc[0],3), 
    round(exp_tonebind_corr['p_val'].iloc[0],4), exp_tonebind_corr['CI95'].iloc[0]
))

No significant correlation between individual expiration ratios and tone binding (r(39)=0.025, p=0.8755, 95% CI [-0.28  0.33]).

In [25]:
# Correlation of inspiration ratio and tone binding
insp_tonebind_corr = pg.corr(x=tone_expinsp_ratio_tonebind_BasTOpT['tone_insp_ratio'], y=tone_expinsp_ratio_tonebind_BasTOpT['tone_binding'])

md("No significant correlation between individual inspiration ratios and tone binding (r({})={}, p={}, 95% CI {}).".format(
    (insp_tonebind_corr['n'].iloc[0]-2), round(insp_tonebind_corr['r'].iloc[0],3), 
    round(insp_tonebind_corr['p_val'].iloc[0],4), insp_tonebind_corr['CI95'].iloc[0]
))

No significant correlation between individual inspiration ratios and tone binding (r(39)=-0.049, p=0.7631, 95% CI [-0.35  0.26]).

In [ ]:
### FIGURE 7d ###
# Correlation plot of tone binding and individual exp/insp ratios

plt.figure(figsize=(6, 6), dpi=300)

# Add horizontal line for group-level mean action binding effect
tonebind_group_avg = -88.18
plt.axhline(tonebind_group_avg, linestyle="dashed", color="k", linewidth=0.8, alpha=0.3)

# Scatter and regression line for tone_exp_ratio
sns.regplot(data=tone_expinsp_ratio_tonebind_BasTOpT, x='tone_exp_ratio', y='tone_binding', marker= 's', 
            scatter_kws={'color': '#8E44AD', 'alpha': 0.7}, line_kws={'color': '#8E44AD'}, label="Expiration Ratio")

# Scatter and regression line for tone_insp_ratio
sns.regplot(data=tone_expinsp_ratio_tonebind_BasTOpT, x='tone_insp_ratio', y='tone_binding',
            scatter_kws={'color': '#8BBB94', 'alpha': 0.7}, line_kws={'color':'#8BBB94'}, label="Inspiration Ratio")

plt.ylabel("Tone Binding: OpT - BasT (ms)", fontsize='x-large')
plt.xlabel("Respiratory phase ratios", fontsize='x-large')
plt.xlim(0.3, 1.4)
plt.legend(loc='upper right')
sns.despine()

# Save plot 
fig10_4 = os.path.join(plotting_dir, "Figure7d_expinsp_ratio_tonebind_correlation.svg")
plt.savefig(fig10_4, format="svg")
plt.show()

## **6. Circular analysis of task events across respiratory cycle [Results 2.3.1]**

This section performs circular analysis of task events around the respiratory cycle. This is reported in Results section 2.3.1, with corresponding Figure 4a and Extended Data Figure 2. In detail, the following steps are included:

- **6a. 1st level circular analysis (within-subject) - individual plots and Rayleigh test**: perform 1st-level circular analysis (within-subject) of task event onset across respiratory cycle. For each participant, the relative onset of each user-defined task event is computed within the respiratory cycle, where events falling in expiration are assigned radian values between 0 and π, where events falling in inspiration are assigned radian values between π and 2π. Then, the participant-specific mean of the circular distribution of the task event onsets is tested against a uniform distribution using a Rayleigh test of uniformity. 
    - If `individual_plots` is set to `True`, individual circular plots with task event onsets and mean exp/insp duration within the respiratory cycle can be generated for each participant and saved in `derivatives/resp-phase-analysis/sub-XX/beh/`.
- **6b. 2nd level circular analysis (between-subjects) - group-level plot and Rayleigh test**: perform 2nd-level circular analysis (between-subject) of task event onset across respiratory cycle. First, the group-level circular mean of task event onsets is derived by averaging the individual circular means, showing the average task event onset in the respiratory cycle across the group, and weighted by its length (mean resultant length ϱ) to reflect the spread of individual means around the circle. Then, the group-level mean resultant length ϱ is tested against the null hypothesis of uniform distribution using a Rayleigh test for uniformity (Pewsey et al., 2013). If ϱ gets sufficiently high to exceed a threshold value (i.e., the set of individual means is not spread evenly across the respiratory cycle), the data can be interpreted as too locally clustered to be consistent with a uniform distribution. 
    - If `group_plot` is set to `True`, group-level circular plot with circular mean and mean resultant length rho, as well as individual means of task event onsets and average exp/insp duration can be generated and saved in `derivatives/resp-phase-analysis/`.
- **6c. Non-parametric boostrapping analysis**: non-parametrically calculate 95% confidence intervals and significance through bootstrapping analysis (Ohl et al., 2016; Kunzendorf et al., 2019). Based on N of participants, a bootstrap sample of same size is drawn (with replacement). For each participant in the bootstrap sample, a circular density of task event onsets is computed and then the mean circular density of all participants in the bootstrap sample is derived. This bootstrap procedure is repeated `n_boot` times (recommended 10000 for full analysis). The 95% confidence intervals are derived as the 2.5% and 97.5% percentiles from the distribution of mean circular densities, so that significant deviations from a circular uniform are determined when the 95% confidence interval is outside the circular density of a uniform distribution.
  - If `perform_bootstrap` is set to `True`, two serialized pickle files are saved in `derivatives/resp-phase-analysis`: `*_resp_2ndlev_circular_bootstrap_{abbrev}.pkl` containing results of each bootstrapping sample, and `*_resp_2ndlev_circular_bootstrap_signif.pkl` containing CI, median, and significance markers
- **6d. Group-level circular plot with bootstrapping CI and significance**: create the group-level circular plot with 2nd level circular mean and mean resultant length rho (central arrow, direction and length respectively), individual circular means of task event onsets (black dots), average circular distribution with 95% CI and significant segments, as determined by bootstrap analysis, colored differently based on average expiration (purple) or inspiration (green) duration.

In [44]:
############## General settings for circular analysis ##############

# Define function to draw segments in circular plot
def mean_circseg(rad):
    mean_rad = circular.mean_circular(rad, na_rm=True)[0]
    xcoord = math.sin(mean_rad)
    ycoord = math.cos(mean_rad)

    return xcoord, ycoord

# Define R-compatible parameters for circular plots
r_pi = robjects.r['pi'] # extract pi value
r_pi = r_pi[0]
rvector_xlim = robjects.FloatVector([-1.3, 1.3])
rvector_ylim = robjects.FloatVector([-1.3, 1.3])

### **6a. 1st level circular analysis (within-subject): individual plots and Rayleigh test**

In [45]:
############## 6a. 1st level circular analysis (within-subject): individual plots and Rayleigh test ##############

# Define function to perform 1st level circular analysis of task event onset across respiratory cycle
def circular_analysis_1stlev(beh_resp_df, abbrev, individual_plots=False, participant_col=column_map['participant'], 
                             respphase_dir=respphase_dir, datatype_name=datatype_name):
    """
    Perform 1st-level circular analysis (within-subject) of task event onset across respiratory cycle. 

    For each participant, RESP-derived respiratory phase data is processed, including the relative onset 
    of the user-defined task event within the respiratory cycle. Then, the participant-specific 
    mean of the circular distribution of the task event onsets is tested against a uniform distribution
    using a Rayleigh test of uniformity. Optionally, individual circular plots with task event onsets 
    can be generated in 'derivatives/resp-phase-analysis/sub-XX/beh'.

    Parameters
    ----------
    beh_resp_df : DataFrame
        DataFrame containing preprocessed behavioral data and relevant RESP features.
    abbrev : str
        Abbreviated name for the event type (e.g., 'act'), used as a suffix for all derived columns.
    individual_plots : bool, default=False
        If True, generates individual circular plots of task event onsets within the respiratory cycle.
    participant_col : str
        Column storing participant IDs.
    respphase_dir : str
        Directory where the respiratory phase analysis results are stored.
    datatype_name : str
        BIDS-compatible datatype folder for individual plot sub-directory.

    Returns
    -------
    dict
        event_resp_raytest : dict containing Rayleigh test results for each participant.
        
        Example structure:
        ```
        {
            participant_id: {
                'circular_mean': float,       # Circular mean (radians)
                'mean_length_rho': float,     # Mean resultant length
                'raytest_statistic': float,   # Rayleigh test statistic
                'raytest_pvalue': float       # p-value of Rayleigh test
            },
            ...
        }
        ```

    """

    # Initialize dict for storing Rayleigh test results for each subj
    event_resp_raytest = {}

    # Loop over each participant
    for subj in beh_resp_df[participant_col].unique():

        # Define directory for storing individual plots of 1st level circ analysis
        if individual_plots:
            indivplot_dir = os.path.join(respphase_dir, f'sub-{subj}', datatype_name)
            if not os.path.exists(indivplot_dir):
                os.makedirs(indivplot_dir)

        # Filter data for the current participant
        subj_df = beh_resp_df[beh_resp_df[participant_col] == subj]
        resp_radian_event = subj_df[f'event_resp_rad_{abbrev}'].dropna() # extract position in rad of task event within resp cycle

        # Convert from pandas to R dataframe
        resp_radian_event_r = ro.conversion.py2rpy(resp_radian_event)
  

        ############## 1st level circular analysis: individual plots ##############

        if individual_plots: 

            # # Expiration onset 
            # exp_onset_rad = 2 * np.pi * beh_resp_df[f'RR_resp_1perSec_{abbrev}'] * 0        # convert in rad (expiration onset is point 0)
            # exp_onset_rad_r = ro.conversion.py2rpy(exp_onset_rad)

            # # Inspiration onset 
            # insp_onset_s = beh_resp_df[f'InspOn_{abbrev}'] - beh_resp_df[f'ExpOn_pre_{abbrev}']         # insp onset between expon_pre and expon_post
            # insp_onset_rad = 2 * np.pi * beh_resp_df[f'RR_resp_1perSec_{abbrev}'] * insp_onset_s        # convert in rad
            # insp_onset_rad_r = ro.conversion.py2rpy(insp_onset_rad)

            # # Define x and y coordinate of mean expiration/inspiration onset
            # exp_onset_xcoord, exp_onset_ycoord = mean_circseg(exp_onset_rad_r)
            # insp_onset_xcoord, insp_onset_ycoord = mean_circseg(insp_onset_rad_r)


            ##### Prepare individual circular plot #####

            # Calculate circular density
            res40 = circular.density_circular(resp_radian_event_r, bw=40)

            # Define directory of plot
            plot_name = f"sub-{subj}_task-CardiAgIBTask_resp_1stlev_circular_{abbrev}.png"
            plot_dir = os.path.normpath(os.path.join(indivplot_dir, plot_name))
            grdevices.png(file=plot_dir, width=2500, height=2500, res=300)

            # Circular plot with data points of task event onset for one participant
            circular.plot_circular(resp_radian_event_r, units="radians", rotation="clock", zero=r_pi/2, stack=True, pch=16, 
                                   cex=1.5, ticks=True, tcl=0.08, col="black", xlim=rvector_xlim, ylim=rvector_ylim)
            
            # # Draw mean onset for expiration/inspiration onset
            # graphics.segments(0,0,exp_onset_xcoord, exp_onset_ycoord, col='#8E44AD', lty=2, lwd=4)       # segment for mean exp onset
            # graphics.segments(0,0,insp_onset_xcoord, insp_onset_ycoord, col='#8BBB94', lty=2, lwd=4)       # segment for mean insp onset
            
            # Draw arrow for circular mean (i.e., direction) and mean resultant length of task event onsets
            circular.arrows_circular(circular.mean_circular(resp_radian_event_r), y=circular.rho_circular(resp_radian_event_r), 
                                     lwd=5, zero=r_pi/2, rotation="clock", cex=1, col="black")       # arrow for circular mean and res. length
            
            # Add Kernel Density Estimate (KDE) of task event onset
            circular.lines_density_circular(res40, lwd=3, lty=2, col="darkgrey", rotation="clock", zero=r_pi/2) # add circular density 
            
            circular.axis_circular(units="radians", rotation="clock", zero=r_pi/2, template="none", cex=1.5)    # draw axes labels 
            grdevices.dev_off()


        ############## 1st level circular analysis: individual Rayleigh test ##############
    
        # Calculate circular mean and mean resultant length of task event onsets across respiratory cycle
        circ_mean_event = circular.mean_circular(resp_radian_event_r)
        circ_rho_event = circular.rho_circular(resp_radian_event_r)

        # Perform Rayleigh test of uniformity on task event onsets across respiratory cycle
        subj_raytesttmp = circular.rayleigh_test(resp_radian_event_r)
        subj_raytest_pval = subj_raytesttmp[1][0]       # p-value of Rayleigh test
        subj_raytest_stat = subj_raytesttmp[0][0]       # mean resultant length (statistics)
        
        # Initialize the dict for the current subject
        if subj not in event_resp_raytest:
            event_resp_raytest[subj] = {}
        
        # Add the result to the dict
        event_resp_raytest[subj] = {'circular_mean': circ_mean_event[0],
                                    'mean_length_rho': circ_rho_event[0],
                                    'raytest_statistic': subj_raytest_stat, 
                                    'raytest_pvalue': subj_raytest_pval}
        
    return event_resp_raytest

In [46]:
# Perform 1st level (within-subject) circular analysis of action onset across respiratory cycle
act_resp_raytest = circular_analysis_1stlev(beh_resp_df=beh_resp_action, 
                                            abbrev=abbrev, individual_plots=False)

In [47]:
# Calculate FDR-corrected within-subject p-values of Rayleigh test
keys = list(act_resp_raytest.keys())
pvals = np.array([act_resp_raytest[subj]['raytest_pvalue'] for subj in keys])

rejected, pvals_fdr = fdrcorrection(pvals, alpha=0.05, method='indep') # apply FDR correction

for i, subj in enumerate(keys):
    act_resp_raytest[subj]['raytest_pvalue_fdr'] = pvals_fdr[i]      # FDR-corrected p-values
    act_resp_raytest[subj]['raytest_significant_fdr'] = rejected[i]  # significance after FDR correction (bool)

significant_count = sum(act_resp_raytest[subj]['raytest_significant_fdr'] for subj in act_resp_raytest)
significant_percent = significant_count / len(act_resp_raytest) * 100

print(f'{significant_count} out of {len(act_resp_raytest)} participants ({round(significant_percent,2)}%) show a significant FDR-corrected within-subject p-value for the Rayleigh test of action onsets across the respiratory cycle.')

39 out of 41 participants (95.12%) show a significant FDR-corrected within-subject p-value for the Rayleigh test of action onsets across the respiratory cycle.


In [48]:
# Specify min and max FDR-corrected p-value range
significant_fdr_values = [
    v['raytest_pvalue_fdr'] 
    for v in act_resp_raytest.values() 
    if v['raytest_significant_fdr']
]

min_fdr = min(significant_fdr_values)
max_fdr = max(significant_fdr_values)

print(f"Range of FDR-corrected p-values (significant only): {min_fdr:.2e} – {max_fdr:.2e}")

Range of FDR-corrected p-values (significant only): 2.44e-31 – 8.27e-03


In [49]:
############## 6b. 2nd level circular analysis (between-subjects): group-level plot and Rayleigh test ##############

# Define function for performing 2nd level circular analysis and, optionally, create group-level plot
def circular_analysis_2ndlev(beh_resp_df, abbrev, event_resp_raytest, group_plot=True, circdens_bw=20, 
                             participant_col=column_map['participant'], plotting_dir=plotting_dir, 
                             save_svg=False, svg_fname=None, condition_plot=False):
    """
    Perform 2nd level circular analysis (between-subjects) of task event onset across respiratory cycle.

    First, the group-level circular mean of task event onsets is derived by averaging the individual circular means, 
    and weighted by its mean resultant length rho to reflect the spread of individual means around the circle. 
    Then, the group-level mean resultant length rho is tested against the null hypothesis of uniform distribution 
    using a Rayleigh test for uniformity (Pewsey et al., 2013). Optionally, the group-level circular plot with individual 
    means of task event onsets, as well as group-level circular mean and mean resultant length and mean sys/dia duration 
    can be generated and saved in `results/datavisualization`.

    Parameters
    ----------
    beh_resp_df : DataFrame
        DataFrame containing preprocessed behavioral data and relevant RESP features.
    abbrev : str
        Abbreviated name for the event type (e.g., 'act'), used as a suffix for all derived columns.
    event_resp_raytest : dict
        Dictionary containing the 1st level (within-subject) circular analysis results, derived from
        the previous section circular_analysis_1stlev(). 
    group_plot : bool, default=True
        If True, generates the group-level circular plot of mean task event onsets within the cardiac cycle, 
        incl. arrow for circular mean and rho, and Kernel Density Estimate. 
    circdens_bw : int
        Smoothing bandwidth value for drawing Kernel Density Estimate (KDE) in plot; recommended to adjust
        according to own data, not too spiky nor too smooth. Default is 20. 
    participant_col : str, default (column_map['participant'])
        Column name storing the participant IDs.   
    plotting_dir : str
        Directory where figures are stored.
    save_svg : bool, default=False
        If True, saves group-level plot as SVG suitable for publication.
    svg_fname : str
        If save_svg is set to True, defines the SVG file name.

    Returns
    -------
    Two dictionaries: 
        - inspon_means_2ndlev: dict containing mean exp/insp onsets and offsets (in rad), both as pd.Series and R objects
        - raytest_2ndlev_res: dict containing 2nd level Rayleigh test results for the entire group, plus a list of all individual
        circular means, as well as group-level circular mean and rho values 

    """ 

    ############## 2nd level circular analysis (between-subjects): plot ##############

    if group_plot: 
        
        # Expiration onset (0 rad)
        exp_onset_rad = np.zeros(len(beh_resp_df))   # always 0
        exp_onset_rad_r = ro.conversion.py2rpy(exp_onset_rad)

        # Inspiration onset (π rad)
        insp_onset_rad = np.full(len(beh_resp_df), np.pi)   # always pi
        insp_onset_rad_r = ro.conversion.py2rpy(insp_onset_rad)
        
        # Save list of circular means for each participant 
        circ_means_act = []

        for subj in beh_resp_df[participant_col].unique():
            subj_circ_mean = event_resp_raytest[subj]['circular_mean']
            circ_means_act.append(subj_circ_mean) # append value of circular mean for action onsets

        # Convert from pandas to R dataframe
        circ_means_act = pd.Series(circ_means_act)
        circ_means_act_r = ro.conversion.py2rpy(circ_means_act)

        # Define x and y coordinate of mean expiration/inspiration onset
        exp_onset_xcoord, exp_onset_ycoord = mean_circseg(exp_onset_rad_r)
        insp_onset_xcoord, insp_onset_ycoord = mean_circseg(insp_onset_rad_r)

        # Save group-level mean onset and offset of and inspiration (in rad) in a dict
        inspon_means_2ndlev = {'mean_expon': exp_onset_rad, 'mean_expon_r': exp_onset_rad_r, 
                               'mean_inspon': insp_onset_rad, 'mean_inspon_r': insp_onset_rad_r}

        ##### Prepare individual circular plot #####
        
        # Calculate circular density
        res_2ndlev = circular.density_circular(circ_means_act_r, bw=int(circdens_bw))

        if save_svg:
            
            # Define directory of 2nd level circular plot (saved as SVG for publication)
            plot_dir_svg = os.path.join(plotting_dir, svg_fname)
            grdevices.svg(file=plot_dir_svg, width=12, height=8) 

        else:              

            # Define directory of 2nd level circular plot in derivatives\cardiac-phase-analysis\ (saved as PNG and shown within notebook)
            plot_2ndlev_dir = os.path.join(plotting_dir, f"2ndlev_circular_resp_{abbrev}.png")
            grdevices.png(file=plot_2ndlev_dir, width=7000, height=5000, res=600)
        
        graphics.par(mar = robjects.FloatVector([0.2, 0.2, 0.2, 0.2]))

        # Circular plot with data points for all participants
        circular.plot_circular(circ_means_act_r, units="radians", rotation="clock", zero=r_pi/2, stack=False, pch=16, 
                               cex=2, ticks=True, tcl=0.08, col='black', axes=False, xlim=rvector_xlim, ylim=rvector_ylim)
        
        if condition_plot: 
            # Draw mean onset for expiration/inspiration onset
            graphics.segments(0,0,exp_onset_xcoord, exp_onset_ycoord, col='#8E44AD', lty=2, lwd=4)          # segment for mean exp onset
            graphics.segments(0,0,insp_onset_xcoord, insp_onset_ycoord, col='#8BBB94', lty=2, lwd=4)        # segment for mean insp onset
        
        # Draw arrow for circular mean (i.e., direction) and mean resultant length (i.e., arrow length) of task event onsets
        circular.arrows_circular(circular.mean_circular(circ_means_act_r), y=circular.rho_circular(circ_means_act_r), 
                                lwd=5, zero=r_pi/2, rotation="clock", cex=1.5, col='black')                           # arrow for circular mean and res. length
        
        # Add Kernel Density Estimate (KDE) of task event onset
        circular.lines_density_circular(res_2ndlev, lwd=3, lty=2, col='darkgrey', rotation="clock", zero=r_pi/2, cex=2)    # add circular density 
        
        circular.axis_circular(units="radians", rotation="clock", zero=r_pi/2, template="none", cex=2)            # draw axes labels 
        grdevices.dev_off()

        if not save_svg: 
            # Load and display the 2nd level circular plot from PNG image
            plot = mpimg.imread(plot_2ndlev_dir)

            plt.figure(figsize=(8, 8))  
            plt.axis('off')  # Hide axes
            plt.imshow(plot)
            
    
    ############## 2nd level circular analysis (between-subjects): Rayleigh test ##############

    # Calculate circular mean and mean resultant length of task event onsets across respiratory cycle for entire group
    circ_mean_2lev = circular.mean_circular(circ_means_act_r)[0]
    circ_rho_2lev = circular.rho_circular(circ_means_act_r)[0]
    circ_sd_2lev = circular.sd_circular(circ_means_act_r)[0]

    # Perform Rayleigh test of uniformity on task event onsets across respiratory cycle for entire group
    raytesttmp_2lev = circular.rayleigh_test(circ_means_act_r)
    raytest_pval_2lev = raytesttmp_2lev[1][0]   # p-value of Rayleigh test
    raytest_stat_2lev = raytesttmp_2lev[0][0]   # mean resultant length (statistics)

    # Save 2nd level Rayleigh test results
    raytest_2ndlev_res = {'individual_circular_means': circ_means_act,  # list of individ. circular means for all participants
                          'circular_mean': circ_mean_2lev,              # group-level circular mean (direction)
                          'mean_length_rho': circ_rho_2lev,             # group-level mean resultant length (rho)
                          'circular_sd': circ_sd_2lev, 
                          'raytest_statistic': raytest_stat_2lev,
                          'raytest_pvalue': raytest_pval_2lev}
    
    print(f'Rayleigh Test of Uniformity: \n' + f'Test statistic: {raytest_stat_2lev}\n' +
          f'p-value: {raytest_pval_2lev}') 

    return raytest_2ndlev_res, inspon_means_2ndlev


In [ ]:
# Perform 2nd level circular analysis of action onset across respiratory cycle
raytest_2ndlev_res, inspon_means_2ndlev = circular_analysis_2ndlev(beh_resp_df=beh_resp_action, abbrev=abbrev, 
                                                                   event_resp_raytest=act_resp_raytest, group_plot=True, 
                                                                   # save_svg=True, svg_fname=f"2ndlev_circular_resp_{abbrev}.svg", 
                                                                   circdens_bw=20)

In [51]:
############## 6c. Non-parametric bootstrapping analysis ##############

# Define function for bootstrapping analysis: non-parametical computation of confidence intervals 
# and significance for circular data (based on Ohl et al., 2016)
def bootstrap_circular(beh_resp_df, abbrev, n_boot, bw_param=20, 
                       participant_col=column_map['participant'], respphase_dir=respphase_dir):
    """
    Perform a non-parametric bootstrap analysis on circular data for computing confidence intervals and significance.
    
    * From the original N of participants, a bootstrap sample of same size is drawn (with replacement)
    * For each participant in the bootstrap sample, a circular density (with specified bandwidth) of task event onsets is computed
    * The mean circular density across all participants in the bootstrap sample is computed
    * This bootstrap procedure is repeated n_boot times (recommended 10000 for full analysis)
    * 95% confidence intervals are computed as 2.5% and 97.5% percentiles from the distribution of mean circular densities. 
    * Significance is determined when the 95% CI is outside the circular density of a uniform distribution.

        Parameters
    ----------
    beh_resp_df : DataFrame
        DataFrame containing preprocessed behavioral data and relevant RESP features.
    abbrev : str
        Abbreviated name for the event type (e.g., 'act'), used as a suffix for all derived columns.
    n_boot : int
        Number of bootstrap samples to be computed (recommended 10000 for full analysis, 100 for testing).
    bw_param : int, default 20
        Bandwidth parameter for Kernel Density Estimation.
    participant_col : str, default (column_map['participant'])
        Column name storing the participant IDs.   
    respphase_dir : str
        Directory where the respiratory phase analysis results are stored.
    
    Returns:
    ----------
       Two serialized .pkl files are saved in the specified respphase_dir: 
        - '*_resp_2ndlev_circular_bootstrap_{abbrev}.pkl' containing results of each bootstrapping sample
        - '*_resp_2ndlev_circular_bootstrap_signif_{abbrev}.pkl' containing CI, median, and significance markers
    

    """
    
    # Extract unique participant list
    subj_list = beh_resp_df[participant_col].unique()
    
    # Output storage
    out = []
    
    # Bootstrap procedure
    for i in tqdm(range(n_boot), desc="Bootstrapping"):
        
        # Draw bootstrap sample of same size as original sample, with replacement 
        subj_list_boot = np.random.choice(subj_list, size=len(subj_list), replace=True)
        buffer = [] # tmp buffer for subj data
        
        for subj in subj_list_boot:
            # Filter data for this participant
            data = beh_resp_df.loc[beh_resp_df[participant_col] == subj, 
                                  f'event_resp_rad_{abbrev}'].values
            data = data[~np.isnan(data)]  # remove NaNs

            if len(data) > 0:
                # Convert participant data to R circular object
                r_data = robjects.FloatVector(data)
                r_circular_data = circular.circular(r_data, type="angles", units="radians", 
                                                    modulo="2pi", zero=r_pi/2, rotation="clock")

                # Compute circular density
                kde_res = robjects.r('density.circular')(r_circular_data, bw=bw_param)
                density_values = np.array(kde_res[2])  # Extract density values
                buffer.append(density_values)
        
        if buffer:
            # Compute mean circular density across participants
            buffer_mean = np.mean(np.array(buffer), axis=0)
            out.append(buffer_mean)
    
    out = np.array(out) # convert to np array

    # Compute 95% confidence intervals and median
    ci_lower = np.percentile(out, 2.5, axis=0)
    ci_upper = np.percentile(out, 97.5, axis=0)
    ci_median = np.percentile(out, 50, axis=0)

    # Determine significant segments where the CI are outside uniform distribution
    markup_upper = np.where(ci_lower > circular.dcircularuniform(1)) # where ci_lower (2.5%) outside uniform -> signif. more event onsets
    markup_lower = np.where(ci_upper < circular.dcircularuniform(1)) # where ci_upper (97.5%) outside uniform -> signif. less event onsets

    # Ensure directory exists
    os.makedirs(respphase_dir, exist_ok=True)

    # Save array with results of each bootstrapping sample
    bootres_dir = os.path.join(respphase_dir, f'task-CardiAgIBTask_resp_2ndlev_circular_bootstrap_{abbrev}.pkl')
    with open(bootres_dir, "wb") as f:
        pickle.dump(out, f)

    # Save dict with CI, median, and significance markers
    bootres_signif_dir = os.path.join(respphase_dir, f'task-CardiAgIBTask_resp_2ndlev_circular_bootstrap_signif_{abbrev}.pkl')
    with open(bootres_signif_dir, "wb") as f:
        pickle.dump({"ci_lower": ci_lower, "ci_upper": ci_upper, "ci_median": ci_median,
                     "markup_upper": markup_upper, "markup_lower": markup_lower}, f)


    return print(f"Bootstrap complete. Results saved in {respphase_dir} folder.")

In [52]:
# Perform the bootstrapping analysis for action onsets
perform_bootstrap = False  # set to True only when the bootstrap analysis is required (might take very long!)

if perform_bootstrap:
    bootstrap_circular(beh_resp_df=beh_resp_action, 
                       abbrev=abbrev, 
                       n_boot=10000,        # should be set to 10000 iterations for actual analyses
                       bw_param=20)

### **6d. Group-level circular plot with bootstrapping CI and significance**

In [55]:
############## 6d. Group-level circular plot with bootstrapping CI and significance ##############

# Define function to plot 2nd level circular analysis results, including: group-level circular mean and 
# mean resultant length rho of task event onsets; average circular distribution with 95% CI and significant 
# segments, as determined by bootstrap analysis; individual points for mean task event onsets
def plot_bootstrap_circular_2ndlev(beh_resp_df, abbrev, participant_col=column_map['participant'], respphase_dir=respphase_dir):

    # Define color palette
    colors_map = {'exp': '#8E44AD', 'insp': '#8BBB94'}

    # Load info about bootstrapping analysis significance from .pkl file (CI, median, signif. segments)
    bootres_signif_dir = os.path.join(respphase_dir, f'task-CardiAgIBTask_resp_2ndlev_circular_bootstrap_signif_{abbrev}.pkl')
    if os.path.exists(bootres_signif_dir): # check if the file exists before opening
        bootres_dict = pd.read_pickle(bootres_signif_dir)  
    else:
        print(f"Warning: The file '{bootres_signif_dir}' does not exist.")
        bootres_dict = None  # Optional: Set to None or handle accordingly

    
    # Normalize CI and median and convert to R objects
    ci_median_r = robjects.FloatVector(bootres_dict['ci_median'] / circular.dcircularuniform(1))
    ci_lower_r = robjects.FloatVector(bootres_dict['ci_lower'] / circular.dcircularuniform(1))
    ci_upper_r = robjects.FloatVector(bootres_dict['ci_upper'] / circular.dcircularuniform(1))

    # Save list of circular means for each participant 
    circ_means = []

    for subj in beh_resp_df[participant_col].unique():
        subj_circ_mean = act_resp_raytest[subj]['circular_mean']
        circ_means.append(subj_circ_mean) # append value of circular mean for action onsets

    # Convert from pandas to R dataframe
    circ_means = pd.Series(circ_means)
    circ_means_r = ro.conversion.py2rpy(circ_means)

    # plot_dir = plotting_dir + f'\\Figure4a_resp_bootstrap_circular_{abbrev}.png'
    # grdevices.png(file=plot_dir, width=7000, height=5000, res=600)
    plot_dir_svg = plotting_dir + f'\\Figure4a_resp_bootstrap_circular_{abbrev}.svg'     # uncomment to save SVG
    grdevices.svg(file=plot_dir_svg, width=12, height=8)                # uncomment to save SVG

    # Generate the sequence of x-values of circular distribution (in rad) and convert into R object
    seq_x = np.linspace(0, 2 * np.pi, num=len((bootres_dict['ci_median'])))
    seq_x_r = robjects.FloatVector(seq_x)

    circular.plot_circular(circ_means_r, units="radians", rotation="clock", zero=r_pi/2, stack=False, pch=16, 
                           cex=1.5, ticks=False, tcl=0.08, col="black", axes=False, xlim=rvector_xlim, ylim=rvector_ylim)
    circular.arrows_circular(circular.mean_circular(circ_means_r), y=circular.rho_circular(circ_means_r), 
                             lwd=5, zero=r_pi/2, rotation="clock", cex=1, col="black")                              # arrow for circular mean and res. length
    circular.axis_circular(units="radians", rotation="clock", zero=r_pi/2, template="none", cex=1.5)                # draw axes labels 
    circular.lines_circular(seq_x_r, ci_lower_r, join=True, col="grey", lwd=1.5, rotation="clock", offset=0, zero=r_pi/2)
    circular.lines_circular(seq_x_r, ci_upper_r, join=True, col="grey", lwd=1.5, rotation="clock", offset=0, zero=r_pi/2)

    # Add mean circular density of bootstrapping sample
    circular.lines_circular(seq_x_r, ci_median_r, join=True, col="grey", lwd=4, rotation="clock", offset=0, zero=r_pi/2)

    # Compute indices for expiration and inspiration segments
    mexppos = np.where((seq_x > 0) & (seq_x <= np.pi))[0]
    minsppos = np.where((seq_x >= np.pi) & (seq_x <= 2*np.pi))[0]

    exp_x = robjects.FloatVector(seq_x[mexppos])
    exp_y = robjects.FloatVector((bootres_dict['ci_median'] / circular.dcircularuniform(1))[mexppos])
    insp_x = robjects.FloatVector(seq_x[minsppos])
    insp_y = robjects.FloatVector((bootres_dict['ci_median'] / circular.dcircularuniform(1))[minsppos])

    # Plot expiration (purple) and inspiration (green) segments
    circular.lines_circular(exp_x, exp_y, join=False, col=colors_map['exp'], lwd=4, rotation="clock", offset=0, zero=r_pi/2)
    circular.lines_circular(insp_x, insp_y, join=False, col=colors_map['insp'], lwd=4, rotation="clock", offset=0, zero=r_pi/2)

    # Extract significant density segments outside upper CI (where significantly more task event onsets)
    markup_upper_xcoord = robjects.FloatVector(seq_x[bootres_dict['markup_upper'][0]]) 
    markup_upper_ycoord = robjects.FloatVector((bootres_dict['ci_median'] / circular.dcircularuniform(1))[bootres_dict['markup_upper'][0]])

    # Compute differences between consecutive x-coordinates
    diff_x = np.diff(markup_upper_xcoord)

    # Define a threshold to detect discontinuities (e.g., a large jump in radians)
    threshold = np.pi / 2  # Adjust if needed

    # Find split points where the jump is too large
    split_indices = np.where(diff_x > threshold)[0] + 1  # +1 to shift to the next element

    # Split the x and y coordinates at these discontinuities
    x_segments = np.split(markup_upper_xcoord, split_indices)
    y_segments = np.split(markup_upper_ycoord, split_indices)

    # Convert expiration and inspiration indices to sets for easy checking
    exp_indices = set(mexppos)
    insp_indices = set(minsppos)

    # Convert each segment into an R FloatVector and plot separately
    for x_seg, y_seg in zip(x_segments, y_segments):
        x_r = robjects.FloatVector(x_seg)
        y_r = robjects.FloatVector(y_seg)

        # Determine if this segment is expiration or inspiration
        segment_indices = set(np.searchsorted(seq_x, x_seg))  # Find corresponding indices in seq_x

        if segment_indices.issubset(exp_indices):  # Entire segment in expiration
            color = colors_map['exp']
        elif segment_indices.issubset(insp_indices):  # Entire segment in inspiration
            color = colors_map['insp']

        # Plot the segment with the chosen color
        circular.lines_circular(x_r, y_r, join=False, col=color, lwd=8, rotation="clock", offset=0, zero=r_pi/2)

        #print(x_r)

    grdevices.dev_off()

    # # Load and display the 2nd level circular plot
    # plot = mpimg.imread(plot_dir)

    # plt.figure(figsize=(8, 8))  
    # plt.axis('off')  # Hide axes
    # plt.imshow(plot)

    print(f"Significant segments: {bootres_dict['markup_upper']}")

In [56]:
### FIGURE 4a ###

# Plot 2nd level circular plot with bootstrap results
plot_bootstrap_circular_2ndlev(beh_resp_df=beh_resp_action, abbrev=abbrev)

Significant segments: (array([  7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,
        20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,
        33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,
        46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,
        59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,  70,  71,
        72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,
        85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
        98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
       111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123,
       124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136,
       137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149,
       150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162,
       163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175,
       176, 177, 178, 179, 180, 181, 182,

### **6e. Condition-wise analysis of circular distributions of action onsets**

In [57]:
##### 6e. Circular analysis of action onsets in BasA #####

beh_resp_action_BasA = beh_resp_action[beh_resp_action['condition'] == 'BasA']
act_resp_raytest_BasA = circular_analysis_1stlev(beh_resp_df=beh_resp_action_BasA, abbrev=abbrev, individual_plots=False)

print('Action onsets in BasA')
# Perform 2nd level circular analysis of action onset across respiratory cycle in BasA condition
raytest_2ndlev_res_BasA, inspon_means_2ndlev_BasA = circular_analysis_2ndlev(beh_resp_df=beh_resp_action_BasA, abbrev=abbrev, 
                                                                             event_resp_raytest=act_resp_raytest_BasA, 
                                                                             group_plot=True, circdens_bw=25, 
                                                                             plotting_dir=plotting_dir, 
                                                                             save_svg=True, svg_fname='ExDFigure2a_resp_circular_act_BasA.svg')

Action onsets in BasA
Rayleigh Test of Uniformity: 
Test statistic: 0.8796343614254951
p-value: 1.558812393388784e-13


In [58]:
##### 6e. Circular analysis of action onsets in OpA #####

beh_resp_action_OpA = beh_resp_action[beh_resp_action['condition'] == 'OpA']
act_resp_raytest_OpA = circular_analysis_1stlev(beh_resp_df=beh_resp_action_OpA, abbrev=abbrev, individual_plots=False)

print('Action onsets in OpA')
# Perform 2nd level circular analysis of action onset across respiratory cycle in OpA condition
raytest_2ndlev_res_OpA, inspon_means_2ndlev_OpA = circular_analysis_2ndlev(beh_resp_df=beh_resp_action_OpA, abbrev=abbrev, 
                                                                           event_resp_raytest=act_resp_raytest_OpA, 
                                                                           group_plot=True, circdens_bw=25, 
                                                                           plotting_dir=plotting_dir, 
                                                                           save_svg=True, svg_fname='ExDFigure2b_resp_circular_act_OpA.svg')

Action onsets in OpA
Rayleigh Test of Uniformity: 
Test statistic: 0.8263283650935177
p-value: 3.298654069135241e-12


In [59]:
##### 6e. Circular analysis of action onsets in OpT #####

beh_resp_action_OpT = beh_resp_action[beh_resp_action['condition'] == 'OpT']
act_resp_raytest_OpT = circular_analysis_1stlev(beh_resp_df=beh_resp_action_OpT, abbrev=abbrev, individual_plots=False)

print('Action onsets in OpT')
# Perform 2nd level circular analysis of action onset across respiratory cycle in OpT condition
raytest_2ndlev_res_OpT, inspon_means_2ndlev_OpT = circular_analysis_2ndlev(beh_resp_df=beh_resp_action_OpT, abbrev=abbrev, 
                                                                           event_resp_raytest=act_resp_raytest_OpT, 
                                                                           group_plot=True, circdens_bw=25, 
                                                                           plotting_dir=plotting_dir, 
                                                                           save_svg=True, svg_fname='ExDFigure2c_resp_circular_act_OpT.svg')

Action onsets in OpT
Rayleigh Test of Uniformity: 
Test statistic: 0.7827326784409733
p-value: 3.1339869859602216e-11


In [60]:
# Watson's two-samples tests for comparisons of action onset distribution across conditions

indiv_means_BasA = ro.conversion.py2rpy(raytest_2ndlev_res_BasA['individual_circular_means'])
indiv_means_OpA = ro.conversion.py2rpy(raytest_2ndlev_res_OpA['individual_circular_means'])
indiv_means_OpT = ro.conversion.py2rpy(raytest_2ndlev_res_OpT['individual_circular_means']) 

watson_test_BasAOpA = circular.watson_two_test(indiv_means_BasA, indiv_means_OpA, alpha=0.01)
watson_test_BasAOpT = circular.watson_two_test(indiv_means_BasA, indiv_means_OpT, alpha=0.01)
watson_test_OpAOpT = circular.watson_two_test(indiv_means_OpA, indiv_means_OpT, alpha=0.01)

watson_res_BasAOpA = str(watson_test_BasAOpA[3])
watson_res_BasAOpT = str(watson_test_BasAOpT[3])
watson_res_OpAOpT = str(watson_test_OpAOpT[3])

print(f"Watson's U2 test - BasA vs. OpA: {watson_res_BasAOpA}")
print(f"Watson's U2 test - BasA vs. OpT: {watson_res_BasAOpT}")
print(f"Watson's U2 test - OpA vs. OpT: {watson_res_OpAOpT}")

Watson's U2 test - BasA vs. OpA: 
      Watson's Two-Sample Test of Homogeneity 

Test Statistic: 0.0716 
Level 0.01 Critical Value: 0.268 
Do Not Reject Null Hypothesis 


Watson's U2 test - BasA vs. OpT: 
      Watson's Two-Sample Test of Homogeneity 

Test Statistic: 0.0692 
Level 0.01 Critical Value: 0.268 
Do Not Reject Null Hypothesis 


Watson's U2 test - OpA vs. OpT: 
      Watson's Two-Sample Test of Homogeneity 

Test Statistic: 0.062 
Level 0.01 Critical Value: 0.268 
Do Not Reject Null Hypothesis 




## **7. Pre- and post-event respiratory cycle changes [Results 2.5.2]**

This section analyzes changes in the duration of respiratory cycles before and after the task event of interest, indicative of respiratory deceleration and/or acceleration. In detail: 

- **7a. Extract respiratory cycles before and after action**: extract R-R intervals before (T-2, T-1) and after (T+1, T+2) the R-R interval where the task event (i.e., action) occurs (T0); in the CardiAg intentional binding task, the event of interest is the action onset, and perform a two-way repeated-measure ANOVA 5 (Timepoint) x 3 (Condition) on R-R interval duration. This is reported in Results section 2.5.2, with corresponding Extended Data Figure 3. 

In [61]:
############## 7a. Extract respiratory cycles before and after action ##############

# Define function for extracting respiratory cycles before (T-2, T-1) and after (T+1, T+2) 
# the respiratory cycle where the task event occurs (T0)
def extract_prepost_rr(beh_resp_df, abbrev, column_map=column_map, subj_resp=subj_resp): 
    
    beh_resp_resp_dur = [] # initialize empty list to store dfs for all participants

    # Extract relevant ECG features for the current participant
    for subj in beh_resp_df[column_map['participant']].unique():
        subj_resp_data = subj_resp.get(f'sub-{subj}', {})
        exp_onsets = subj_resp_data.get('exp_onsets_s', [])

        # Filter df with preprocessed beh data for current participant
        subj_resp_dur_df = beh_resp_df[beh_resp_df[column_map['participant']] == subj].copy()

        # Specify relevant column names for ecg-beh analysis based on the event abbreviation
        columns = {name: f"{name}_{abbrev}" for name in [
            'ExpOn_Tminus2', 'ExpOn_Tminus1', 'ExpOn_Tplus1', 'ExpOn_Tplus2',
            'resp_cycle_dur_Tminus2', 'resp_cycle_dur_Tminus1', 'resp_cycle_dur_Tplus1', 'resp_cycle_dur_Tplus2'
        ]}

        subj_resp_dur_df = subj_resp_dur_df.assign(**{col: np.nan for col in columns.values()})

        # Iterate through each row to extract/calculate various ECG features relative to the task event
        for index, row in subj_resp_dur_df.iterrows():
            expon_pre = row[f'ExpOn_pre_{abbrev}']      # specify column with timestamps for the Exp onset pre (just before task event onset)
            expon_post = row[f'ExpOn_post_{abbrev}']    # specify column with timestamps for the Exp onset post (just after task event onset)
            
            # Find the two Exp onsets before the respiratory cycle of task event onset (T-2, T-1)
            expon_Tminus1 = max([exp_m1 for exp_m1 in exp_onsets if exp_m1 < expon_pre], default=np.nan)
            expon_Tminus2 = max([exp_m2 for exp_m2 in exp_onsets if exp_m2 < expon_Tminus1], default=np.nan)
            subj_resp_dur_df.at[index, columns['ExpOn_Tminus1']] = expon_Tminus1       # Exp onset at T-1 before respiratory cycle of task event onset
            subj_resp_dur_df.at[index, columns['ExpOn_Tminus2']] = expon_Tminus2       # Exp onset at T-2 before respiratory cycle of task event onset

            # Find the three Exp onsets after the respiratory cycle of task event onset (T+1, T+2)
            expon_Tplus1 = min([peak_p1 for peak_p1 in exp_onsets if peak_p1 > expon_post], default=np.nan)
            expon_Tplus2 = min([peak_p2 for peak_p2 in exp_onsets if peak_p2 > expon_Tplus1], default=np.nan)
            subj_resp_dur_df.at[index, columns['ExpOn_Tplus1']] = expon_Tplus1       # Exp onset at T+1 after respiratory cycle of task event onset
            subj_resp_dur_df.at[index, columns['ExpOn_Tplus2']] = expon_Tplus2       # Exp onset at T+2 after respiratory cycle of task event onset

            # Calculate respiratory cycles before (T-2, T-1) and after (T+1, T+2, T+3)
            subj_resp_dur_df.at[index, columns['resp_cycle_dur_Tminus2']] = round((expon_Tminus1 - expon_Tminus2),3)
            subj_resp_dur_df.at[index, columns['resp_cycle_dur_Tminus1']] = round((expon_pre - expon_Tminus1),3)
            subj_resp_dur_df.at[index, columns['resp_cycle_dur_Tplus1']] = round((expon_Tplus1 - expon_post),3)
            subj_resp_dur_df.at[index, columns['resp_cycle_dur_Tplus2']] = round((expon_Tplus2 - expon_Tplus1),3)

        beh_resp_resp_dur.append(subj_resp_dur_df)

    # Concatenate the dfs for all participants
    beh_resp_respdur_prepost = pd.concat(beh_resp_resp_dur, ignore_index=True)
    beh_resp_respdur_prepost.rename(columns={'Unnamed: 0': 'subj_index'}, inplace=True)

    # Create df in wide format with mean respiratory cycles pre- and post-task event
    respdur_prepost_wide = beh_resp_respdur_prepost.groupby(['subjID'])[[f'resp_cycle_dur_Tminus2_{abbrev}', f'resp_cycle_dur_Tminus1_{abbrev}', 
                                                              f'resp_cycle_dur_{abbrev}', f'resp_cycle_dur_Tplus1_{abbrev}', 
                                                              f'resp_cycle_dur_Tplus2_{abbrev}']].mean()
    
    # Create df in long format with respiratory cycles duration for each trial and timepoint
    respdur_prepost_long = beh_resp_respdur_prepost.melt(id_vars=[col for col in column_map.values()], 
                                              value_vars=['resp_cycle_dur_Tminus2_act', 'resp_cycle_dur_Tminus1_act', 'resp_cycle_dur_act', 
                                                          'resp_cycle_dur_Tplus1_act', 'resp_cycle_dur_Tplus2_act'],
                                              var_name='Timepoint', value_name='Resp_Cycle_Dur')
    # Replace timepoint names with shorter labels
    timepoint_labels = {
        f'resp_cycle_dur_Tminus2_{abbrev}': 'T-2', f'resp_cycle_dur_Tminus1_{abbrev}': 'T-1', f'resp_cycle_dur_{abbrev}': 'T0',
        f'resp_cycle_dur_Tplus1_{abbrev}': 'T+1', f'resp_cycle_dur_Tplus2_{abbrev}': 'T+2'}
    respdur_prepost_long['Timepoint'] = respdur_prepost_long['Timepoint'].replace(timepoint_labels)
    
    respdur_prepost_long = respdur_prepost_long.dropna()  # drop NaNs of BasT condition and NoResp trials


    return beh_resp_respdur_prepost, respdur_prepost_wide, respdur_prepost_long

In [62]:
# Subselect only relevant columns fro beh-ecg df 
beh_resp_action_rr = beh_resp_action[['subjID', 'condition', 'n_block', 'n_trial', 'real_report_diff_act_deg', 'JE_act',
                                      'keypress_onset', 'ExpOn_pre_act', 'ExpOn_post_act', 'resp_cycle_dur_act', 'act_exp', 'act_insp']].copy()

# Extract respiratory cycle before (T-2, T-1) and after (T+1, T+2) the R-R interval of the action (T0)
beh_resp_respdur_prepost, respdur_prepost_wide, respdur_prepost_long = extract_prepost_rr(beh_resp_df=beh_resp_action_rr, abbrev=abbrev)

In [63]:
# Define function for plotting changes in R-R intervals before and after the task event (T0)
# separately per each task condition

def plot_prepost_rr(respdur_prepost_long, cond_colors, errorbar_type='CI', n_boot=1000, 
                    condition_col=column_map['condition']): 

    # Convert Resp_Cycle_Dur from sec to msec
    respdur_prepost_long['Resp_Cycle_Dur_ms'] = respdur_prepost_long['Resp_Cycle_Dur'] * 1000  # Convert to milliseconds

    # Plot the group average with confidence intervals
    plt.figure(figsize=(8, 5), dpi=300)

    # Extract errorbar type based on user input
    errorbar_arg = ('ci',95) if errorbar_type.lower() == 'CI' else ('se',2)

    # Plot lines for RR interval changes by condition
    sns.lineplot(data=respdur_prepost_long, x='Timepoint', y='Resp_Cycle_Dur_ms', hue=condition_col,
                sort= False, marker='o', linewidth=2, palette=cond_colors, 
                err_style='band', errorbar=errorbar_arg, n_boot=n_boot)

    plt.xlabel("Timepoint relative to keypress onset (T0)")
    plt.ylabel("Mean Respiratory Cycle (ms)")
    plt.xticks(ticks=[0,1,2,3,4], labels=['T-2', 'T-1', 'T0', 'T+1', 'T+2'])
    plt.ylim(3500,4200)
    sns.despine()

    # Save plot 
    figs4 = os.path.join(plotting_dir, "ExDFigure3_resp_changes_prepostaction.svg")
    plt.savefig(figs4, format="svg")

    plt.show()

In [ ]:
### EXTENDED DATA FIGURE 3 ###

# Define color palette for IB task conditions
cond_colpalette = {'BasA': '#a4cbb6', 'OpA': '#007a4e', 'OpT': '#0b3142'}
 
# Plot pre- and post-event R-R interval changes per condition
plot_prepost_rr(respdur_prepost_long=respdur_prepost_long, cond_colors=cond_colpalette, errorbar_type='se')

In [65]:
# Explicitly set Timepoint and Condition columns as categorical, with T0 and BasA as references
respdur_prepost_long['Timepoint'] = pd.Categorical(respdur_prepost_long['Timepoint'], 
                                              categories=['T0', 'T-1', 'T-2', 'T+1', 'T+2'], ordered=True)
respdur_prepost_long['condition'] = pd.Categorical(respdur_prepost_long['condition'], 
                                              categories=['BasA', 'OpA', 'OpT'], ordered=True)

In [ ]:
### Two-way repeated-measure ANOVA 5 (Timepoint) x 3 (Condition) on respiratory cycle duration ###

# Aggregate by participant, timepoint and condition 
respdur_prepost_mean = (
    respdur_prepost_long.groupby(['subjID', 'Timepoint', 'condition'], as_index=False)['Resp_Cycle_Dur'].mean())

# Run rmANOVA
respdur_prepost_anova = pg.rm_anova(
    dv='Resp_Cycle_Dur',
    within=['Timepoint', 'condition'],
    subject='subjID',
    data=respdur_prepost_mean,
    detailed=True)

print(respdur_prepost_anova)

In [67]:
# Pairwise comparisons for Timepoint
timepoint_posthoc = pg.pairwise_tests(dv="Resp_Cycle_Dur", within=["Timepoint"],
                                       subject="subjID", data=respdur_prepost_long,
                                       parametric=True, padjust="bonf")

# Display results
print(timepoint_posthoc)

    Contrast    A    B  Paired  Parametric         T   dof alternative  \
0  Timepoint   T0  T-1    True        True  6.696494  40.0   two-sided   
1  Timepoint   T0  T-2    True        True  5.512599  40.0   two-sided   
2  Timepoint   T0  T+1    True        True  6.679010  40.0   two-sided   
3  Timepoint   T0  T+2    True        True  7.490273  40.0   two-sided   
4  Timepoint  T-1  T-2    True        True -1.509459  40.0   two-sided   
5  Timepoint  T-1  T+1    True        True -0.052170  40.0   two-sided   
6  Timepoint  T-1  T+2    True        True  0.526886  40.0   two-sided   
7  Timepoint  T-2  T+1    True        True  3.039002  40.0   two-sided   
8  Timepoint  T-2  T+2    True        True  2.702473  40.0   two-sided   
9  Timepoint  T+1  T+2    True        True  0.491317  40.0   two-sided   

          p_unc        p_corr p_adjust       BF10    hedges  
0  4.974309e-08  4.974309e-07     bonf  2.825e+05  0.312600  
1  2.285624e-06  2.285624e-05     bonf   7851.921  0.256764  